# CLAUDIN-18.2 meta-analysis

# Setting up

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from typing import Any
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.meta_analysis import combine_effects
from statsmodels.api import OLS, add_constant
from statsmodels.stats.proportion import proportion_confint
from scipy import stats
from scipy.special import logit, expit
from IPython.display import display

df = pd.read_excel(r"PATH/TO/DATA.xlsx", header=1)

print ("\n For Cross-cancer only:\n")

# Adding Sample_dSize:
df["combined_sample_size_used_for_discordance"] = pd.to_numeric(
    df["combined_sample_size_used_for_discordance"],
    errors="coerce")

# Median sample size within each heterogeneity type
group_median = df.groupby("Heterogeneity Type")[
    "combined_sample_size_used_for_discordance"
].transform("median")

display(group_median.value_counts())

group_medians = (
    df.groupby("Heterogeneity Type")[
        "combined_sample_size_used_for_discordance"
    ]
    .median()
)
print(f"The medians for the cross-cancer cohort are for intra- and inter- discordance: {group_medians}")

# 1 if >= group median, 0 if < group median

df["Sample_dSize"] = (df["combined_sample_size_used_for_discordance"] >= group_median).astype("Int64")

# keep missing sample sizes as missing
df.loc[df["combined_sample_size_used_for_discordance"].isna(), "Sample_dSize"] = pd.NA

# LABELS: 
HETEROGENEITY_LABELS = ("Intra-tumoural (T-T)", "Inter-tumoural (T-T)")

STRATIFICATIONS: dict[str, dict[str, Any]] = {
    "Asian_Cohort": {"mapping": {1: "Asian cohorts", 0: "Non-Asian cohorts"}, "mode": "both"},
    "Gastric_Cohort": {"mapping": {1: "Gastric", 0: "Non-gastric"}, "mode": "both"},
    "Stage_IV": {"mapping": {1: "Stage IV", 0: "Stage I-III"}, "mode": "both"},
    "GSample_dSize": {"mapping": {1: "≥ median sample size", 0: "< median sample size"}, "mode": "both"},
    "Majority_G3": {"mapping": {1: "Majority G3", 0: "Majority <G3"}, "mode": "both"},
    "Lymph": {"mapping": {1: "Lymph node", 0: "Non-lymph node"}, "mode": "both"},
    "Peritoneal": {"mapping": {1: "Peritoneal", 0: "Non-peritoneal"}, "mode": "both"},
    "Liver": {"mapping": {1: "Liver", 0: "Non-liver"}, "mode": "both"},
    "Technique": {"mapping": {1: "IHC", 0: "IHC & TMA"}, "mode": "both"},
    "assay_cate_new": {"mapping": {1: "RxDx", 0: "LDT"}, "mode": "both"},
    "automation_re-ex": {"mapping": {1: "Automated", 0: "Manual"}, "mode": "both"},
    "Investigation_Bias": {"mapping": {1: "Bias; only positive cases checked", 0: "No bias"}, "mode": "both"},
    "Threshold": {"mapping": {1: ">75%", 0: "Other"}, "mode": "both"},
    "Cohort_Naive_b": {"mapping": {1: "Treatment-naïve", 0: "Previously treated"}, "mode": "both"},
     "risk_of_bias_patient_selection": {
         "mapping": {1: "High risk", 0: "Low risk", 2: "Unclear"},
         "mode": "both",
     },
     "risk_of_bias_index_test": {
         "mapping": {1: "High risk", 0: "Low risk", 2: "Unclear"},
         "mode": "both",
     },
     "risk_of_bias_flow_timing": {
         "mapping": {1: "High risk", 0: "Low risk", 2: "Unclear"},
         "mode": "both",
     },
}

country_order = ["South Korea","Japan", "Italy", "Germany", "USA"]
    
color_map = {
        "South Korea": "#4E79A7",
        "Italy": "#F28E2B",
        "Japan": "#59A14F",
        "USA": "#E15759",
        "Germany": "#B07AA1",
    }

# TWO cohorts:
df0 = df.copy() # cross-cancer cohort
gastric_only = df[df['Gastric_Cohort']==1].copy() # gastric only cohort

# Gastric-only 

In [ ]:
print ("\n For gastric only:\n")
Gastric_group_medians = gastric_only.groupby("Heterogeneity Type")["combined_sample_size_used_for_discordance"].transform("median")
display(Gastric_group_medians.value_counts())
print('\n')
gastric_only["GSample_dSize"] = (gastric_only["combined_sample_size_used_for_discordance"] >= Gastric_group_medians).astype("Int64")
print(gastric_only.groupby("Heterogeneity Type")['GSample_dSize'].value_counts())

## Study descriptives (GASTRIC)

In [ ]:
results = {}

fig_dir = r"PATH/FOLDER"
os.makedirs(fig_dir, exist_ok=True)

# Overall unique papers
unique_papers = gastric_only["article nr"].nunique()
results["Overall (g) unique nr of papers"] = unique_papers

# Unique paper counts per heterogeneity type
paper_counts = (
    gastric_only
    .groupby("Heterogeneity Type")["article nr"]
    .nunique()
)
results["Paper Split"] = paper_counts

# Overlap: papers that have both intra and inter heterogeneity rows
intra_papers = set(
    gastric_only.loc[
        gastric_only["Heterogeneity Type"] == "Intra-tumoural (T-T)",
        "article nr"
    ].dropna()
)
inter_papers = set(
    gastric_only.loc[
        gastric_only["Heterogeneity Type"] == "Inter-tumoural (T-T)",
        "article nr"
    ].dropna()
)
overlap_papers = intra_papers & inter_papers
results["Papers with both intra and inter"] = len(overlap_papers)
results["Overlapping paper article nrs"] = sorted(overlap_papers)

# Numeric patient col
gastric_only["combined_sample_size_used_for_discordance"] = pd.to_numeric(
    gastric_only["combined_sample_size_used_for_discordance"],
    errors="coerce"
)
patient_nr_col = 'combined_sample_size_used_for_discordance'

# Patient counts: Overall vs split per heterogeneity type
overall_patients = gastric_only[patient_nr_col].sum()
results["Overall (g) patients"] = int(overall_patients)

# Patient counts and distribution per heterogeneity type
patients_split = (
    gastric_only
    .groupby("Heterogeneity Type")[patient_nr_col]
    .agg(
        total="sum",
        median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        min="min",
        max="max",
        n_rows="count"
    )
)
patients_split["IQR"] = patients_split["Q3"] - patients_split["Q1"]
results["Total (g) patients split"] = patients_split

# Cohort collection
def year_range(series):
    years = pd.to_numeric(series, errors="coerce").dropna()
    if years.empty:
        return "NR"
    return f"{int(years.min())} - {int(years.max())}"

def cohort_collection_range(series):
    year_ranges = (
        series
        .astype(str)
        .str.extract(r"(?P<start_year>\d{4})\s*[-–]\s*(?P<end_year>\d{4})")
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
    )
    if year_ranges.empty:
        return "NR"
    start_year = int(year_ranges["start_year"].min())
    end_year = int(year_ranges["end_year"].max())
    return f"{start_year} - {end_year}"
    
# --- Publication years: overall ---
results["Publication years range overall"] = year_range(gastric_only["Year"])

# --- Publication years: per heterogeneity type ---
publication_years_split = (
    gastric_only
    .groupby("Heterogeneity Type")["Year"]
    .apply(year_range, include_groups=False)
)
results["Publication years range split"] = publication_years_split

# --- Cohort collection years: overall ---
results["Cohort collection range overall"] = cohort_collection_range(
    gastric_only["Cohort date range"])

# --- Cohort collection years: per heterogeneity type ---
cohort_collection_split = (
    gastric_only
    .groupby("Heterogeneity Type")["Cohort date range"]
    .apply(cohort_collection_range, include_groups=False)
)
results["Cohort collection range split"] = cohort_collection_split

# Country distribution
def plot_country_pie(data, label, fig_dir):
    country_counts = data["Country"].value_counts(dropna=True)
    results_key = f"Country counts - {label}"
    results[results_key] = country_counts

    if country_counts.empty:
        print(f"No country data available for {label}")
        return
 
    ordered_countries = [c for c in country_order if c in country_counts.index]
    other_countries = [c for c in country_counts.index if c not in country_order]
    country_counts = country_counts.loc[ordered_countries + other_countries]

    country_colors = [
        color_map.get(country, "#CCCCCC")
        for country in country_counts.index
    ]

    plt.figure(figsize=(8, 8))
    country_counts.plot.pie(
        autopct="%1.1f%%",
        colors=country_colors,
        startangle=90,
        wedgeprops={"linewidth": 1, "edgecolor": "white"}
    )
    plt.ylabel("")
    plt.title(f"Country Distribution - {label}")
    plt.tight_layout()
    filename = label
    plt.savefig(
        os.path.join(fig_dir, f"{filename}_country_dist_piechart.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.savefig(
        os.path.join(fig_dir, f"{filename}_country_dist_piechart.svg"),
        bbox_inches="tight"
    )
    plt.close()

# Overall gastric cohort
plot_country_pie(
    gastric_only,
    label="Overall gastric cohort",
    fig_dir=fig_dir)

# Per heterogeneity type
for label, data in gastric_only.groupby("Heterogeneity Type"):
    plot_country_pie(
        data,
        label=label,
        fig_dir=fig_dir    )

# --- Treatment history ---
def treatment_history_counts(data):
    counts = data["Cohort_Naive_b"].value_counts(dropna=False)
    return pd.Series({
        "treatment_naive": counts.get(1, 0),
        "not_treatment_naive": counts.get(0, 0),
        "NR_treatment_naive": counts.get("NR", 0),
        "missing_treatment_naive": data["Cohort_Naive_b"].isna().sum()})

# Overall gastric cohort
treatment_overall = treatment_history_counts(gastric_only)
results["Treatment history overall"] = treatment_overall
# Per heterogeneity type
treatment_split = (
    gastric_only
    .groupby("Heterogeneity Type")
    .apply(treatment_history_counts, include_groups=False))

results["Treatment history split"] = treatment_split

# Gender distribution
female_col = "Females (n)"
male_col = "males_(n)_\ncomputed"
female_pct_col = "Females (%) \ncomputed"
mf_ratio_col = "male:1female_ratio_\ncomputed"

def to_numeric_clean(series):
    return pd.to_numeric(
        series.astype("string").replace(["NR", "n.a", "n.a.", "", " "], pd.NA),
        errors="coerce")

def gender_distribution(data):
    females = to_numeric_clean(data[female_col])
    males = to_numeric_clean(data[male_col])
    female_pct_reported = to_numeric_clean(data[female_pct_col])
    mf_ratio = to_numeric_clean(data[mf_ratio_col])

    total_females = females.sum(skipna=True)
    total_males = males.sum(skipna=True)
    total_known = total_females + total_males

    female_pct = (total_females / total_known) * 100 if total_known > 0 else pd.NA
    male_pct = (total_males / total_known) * 100 if total_known > 0 else pd.NA

    return pd.Series({
        "female_n": int(total_females),
        "male_n": int(total_males),
        "total_with_sex_reported": int(total_known),
        "female_pct": female_pct,
        "male_pct": male_pct,
        "median_female_pct_across_cohorts": female_pct_reported.median(),
        "median_male_to_female_ratio": mf_ratio.median(),
        "n_cohorts_with_gender_data": pd.concat([females, males], axis=1).dropna().shape[0]
    })
        
gender_overall = gender_distribution(gastric_only)
results["Gender distribution overall"] = gender_overall

gender_split = (
    gastric_only
    .groupby("Heterogeneity Type")
    .apply(gender_distribution, include_groups=False))

results["Gender distribution split"] = gender_split

# --- Age distribution ---
age_col = "Cohort Age (median or mean)"
age_range_col = "Corhort Age (range)"  # excel spelling is wrong lol

def age_distribution(data):
    ages = pd.to_numeric(
        data[age_col].astype("string").replace(["NR", "n.a", "n.a.", "", " "], pd.NA),
        errors="coerce")

    age_ranges = (
        data[age_range_col]
        .astype(str)
        .str.extract(r"(?P<age_min>\d+(?:\.\d+)?)\s*[-–]\s*(?P<age_max>\d+(?:\.\d+)?)") #capture this part and name it minimum_age \ one or more digits
        .apply(pd.to_numeric, errors="coerce"))

    valid_ranges = age_ranges.dropna()

    if valid_ranges.empty:
        overall_age_range = "NR"
        age_min = pd.NA
        age_max = pd.NA
    else:
        age_min = valid_ranges["age_min"].min()
        age_max = valid_ranges["age_max"].max()
        overall_age_range = f"{age_min:.0f} - {age_max:.0f}"

    return pd.Series({
        "median_age_across_cohorts": ages.median(),
        "age_min": age_min,
        "age_max": age_max,
        "age_range": overall_age_range,
        "n_cohorts_with_median_age": ages.notna().sum(),
        "n_cohorts_with_age_range": len(valid_ranges)
    })

age_overall = age_distribution(gastric_only)
results["Age distribution overall"] = age_overall

age_split = (
    gastric_only
    .groupby("Heterogeneity Type")
    .apply(age_distribution, include_groups=False))

results["Age distribution split"] = age_split

# Assay Category
assay_col = "assay_cate_new"

def assay_category_counts(data):
    assay = data[assay_col].replace({
        1: "RxDx",
        0: "LDT",
        "1": "RxDx",
        "0": "LDT"})

    counts = assay.value_counts(dropna=False)

    return pd.Series({
        "RxDx": counts.get("RxDx", 1),
        "LDT": counts.get("LDT", 0),
        "NR": counts.get("NR", pd.NA),
    })
assay_overall = assay_category_counts(gastric_only)
results["Assay category overall"] = assay_overall

assay_split = (
    gastric_only
    .groupby("Heterogeneity Type")
    .apply(assay_category_counts, include_groups=False))

results["Assay category split"] = assay_split

# --- Sample type: article split by heterogeneity type ---
sample_type_col = "Sample Type for cohort row"
study_id_col = "article nr"

sample_type_article_counts_split = (
    gastric_only
    .assign(
        sample_type=gastric_only[sample_type_col]
        .astype("string")
        .str.strip()
        .replace(["", "NR", "n.a", "n.a.", "nan"], pd.NA)
        .fillna("NR")
    )
    .groupby(["Heterogeneity Type", "sample_type"])[study_id_col]
    .nunique()
    .reset_index(name="n_articles")
    .sort_values(["Heterogeneity Type", "n_articles"], ascending=[True, False]))

results["Sample type article counts split"] = sample_type_article_counts_split

# --- Threshold / cut-off: article split by heterogeneity type ---
threshold_col = "Threshold"

threshold_article_counts_split = (
    gastric_only
    .assign(
        threshold=gastric_only[threshold_col]
        .astype("string")
        .str.strip()
        .replace(["", "NR", "n.a", "n.a.", "nan"], pd.NA)
        .fillna("NR")
    )
    .groupby(["Heterogeneity Type", "threshold"])[study_id_col]
    .nunique()
    .reset_index(name="n_articles")
    .sort_values(["Heterogeneity Type", "n_articles"], ascending=[True, False]))

results["Threshold article counts split"] = threshold_article_counts_split

# save descriptives table
output_file = os.path.join(fig_dir, "descriptive_results.xlsx")

with pd.ExcelWriter(output_file) as writer:
    for key, value in results.items():
        if isinstance(value, pd.DataFrame):
            df_out = value
        elif isinstance(value, pd.Series):
            df_out = value.reset_index()
            df_out.columns = ["Variable", "Value"]
        else:
            df_out = pd.DataFrame({
                "Variable": [key],
                "Value": [value]
            })

        sheet_name = str(key)[:31]
        df_out.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Results saved to: {output_file}")

## Pooled discordance (GASTRIC)

In [ ]:
def pooled_disc_and_forest_funnel_plots(label='Overall'):
    
    output_dir = r"PATH\FOLDER"
    os.makedirs(output_dir, exist_ok=True)
        
    data = gastric_only.copy()
    

    data["Category"] = data["Heterogeneity Type"]
    
    data["n_samplesize"] = pd.to_numeric(
        data["combined_sample_size_used_for_discordance"],
        errors="coerce")
    
    data["n_discordant"] = pd.to_numeric(
        data["Combined_col_intra_inter discordance rate(n)"],
        errors="coerce")
    
    data = data.dropna(subset=["Category", "n_samplesize", "n_discordant"]).copy()
    
    data["n_samplesize"] = data["n_samplesize"].astype(int)
    data["n_discordant"] = data["n_discordant"].astype(int)
    data["Percentage"] = data["n_discordant"] / data["n_samplesize"]
    
    # ---------------------------------------------------------------------------
    # Meta-analysis function (logit + Wilson optional export)
    # ---------------------------------------------------------------------------
    
    def perform_meta_analysis(subset, category):
        """
        Harmonized random-effects meta-analysis of proportions:
          - Study-level and pooled estimates computed on the logit scale; a 0.5 continuity correction is applied only to proportions of 0 or 1
          - Random-effects pooling via Paule–Mandel (method_re="iterated")
          - Hartung–Knapp–Sidik–Jonkman (HKSJ) CIs for pooled random-effects estimates
          - Wilson CIs exported for reference (not used for pooling)
        """
        # Counts
        x = subset['n_discordant'].to_numpy(dtype=float)
        n = subset['n_samplesize'].to_numpy(dtype=float)
    
        # ---------- Study-level point estimates ----------
        p_raw = x / n
    
        # ---------- Optional: Wilson CIs (for export only) ----------
        wilson_lo, wilson_hi = proportion_confint(
            count=x.astype(int), nobs=n.astype(int), alpha=0.05, method="wilson"
        )
        wilson_lo = np.clip(wilson_lo, 0, 1)
        wilson_hi = np.clip(wilson_hi, 0, 1)
    
        # ---------- Logit transform with boundary-only continuity correction ----------
        # Add 0.5 to the event and non-event counts only when x = 0 or x = n,
        # because the uncorrected logit and its sampling variance are then undefined.
        if np.any(n <= 0):
            raise ValueError("All sample sizes must be positive.")
        if np.any((x < 0) | (x > n)):
            raise ValueError("Discordant counts must satisfy 0 <= x <= n.")

        boundary_mask = (x == 0) | (x == n)
        x_adj = x.copy()
        n_adj = n.copy()
        x_adj[boundary_mask] += 0.5
        n_adj[boundary_mask] += 1.0
        p_adj = x_adj / n_adj
    
        y = logit(p_adj)
        v = 1.0/x_adj + 1.0/(n_adj - x_adj)
        se = np.sqrt(v)
    
        # ---------- Logit-based study-level CIs (display only) ----------
        ci_lo_logit = expit(y - 1.96 * se)
        ci_hi_logit = expit(y + 1.96 * se)
    
        # ---------- Random-effects pooling on logit scale ----------
        # Paule–Mandel estimator for tau² + HKSJ CIs
        res = combine_effects(y, v, method_re="iterated", use_t=True)
        use_t_flag = True
    
        mu_hat = res.mean_effect_re  # pooled *logit*
    
        # conf_int returns:
        # 0: FE (scale=1), 1: RE (scale=1), 2: FE + HKSJ, 3: RE + HKSJ
        ci_fe, ci_re, ci_fe_hksj, ci_re_hksj = res.conf_int(alpha=0.05, use_t=True)
        ci_re_lo, ci_re_hi = ci_re_hksj  # random-effects + HKSJ
    
        # back-transform pooled to %
        pooled_pct    = float(np.clip(expit(mu_hat)   * 100.0, 0, 100))
        pooled_lo_pct = float(np.clip(expit(ci_re_lo) * 100.0, 0, 100))
        pooled_hi_pct = float(np.clip(expit(ci_re_hi) * 100.0, 0, 100))
    
        # ---------- Heterogeneity ----------
        hom = res.test_homogeneity()
        q, df, p_het = hom.statistic, hom.df, hom.pvalue
        tau2 = res.tau2
        I2 = max(0, (q - df) / q) * 100.0 if q > 0 else 0.0
    
        # ---------- Prediction interval (logit scale) ----------
        k = len(y)
        df_pred = max(k - 2, 1)  # conservative df for PI
    
        # HKSJ variance for the pooled RE effect:
        se_pooled_hksj = np.sqrt(res.var_hksj_re)
    
        # Prediction SE includes between-study variance:
        se_pred = np.sqrt(se_pooled_hksj**2 + tau2)
        t_crit = stats.t.isf(0.025, df_pred)
    
        pi_lo_logit = mu_hat - t_crit * se_pred
        pi_hi_logit = mu_hat + t_crit * se_pred
    
        pi_lo_pct = float(np.clip(expit(pi_lo_logit) * 100.0, 0, 100))
        pi_hi_pct = float(np.clip(expit(pi_hi_logit) * 100.0, 0, 100))
    
        # ---------- Egger’s regression test (logit scale) ----------
        z = y / se
        X = add_constant(1.0 / se)
        egger_model = OLS(z, X).fit()
        egger_p = egger_model.pvalues[0]
    
        # ---------- DataFrames for export ----------
        study_level_df = pd.DataFrame({
            'Category': category,
            'Study': subset['Study'].values,
            'Sample Size': n.astype(int),
            'Events': x.astype(int),
            'Effect Size (proportion)': p_raw,
            'CI Lower (Logit 95%)': ci_lo_logit,
            'CI Upper (Logit 95%)': ci_hi_logit,
            'CI Lower (Wilson 95%)': wilson_lo,
            'CI Upper (Wilson 95%)': wilson_hi,
            'Effect (logit)': y,
            'SE (logit)': se
        })
    
        meta_summary_df = pd.DataFrame({
            'Category': [category],
            'Pooled % (Random Effects, logit-normal)': [pooled_pct],
            'CI Lower % (95%)': [pooled_lo_pct],
            'CI Upper % (95%)': [pooled_hi_pct],
            'Prediction Interval Lower % (95%)': [pi_lo_pct],
            'Prediction Interval Upper % (95%)': [pi_hi_pct],
            'Between-Study Variance (tau²)': [tau2],
            'Q-Statistic': [q],
            'df': [df],
            'p-value for Heterogeneity': [p_het],
            'I² (%)': [I2],
            'Knapp–Hartung used': [use_t_flag],
            'Egger intercept p-value (logit)': [egger_p],
            'Total Sample Size': [int(n.sum())],
            'Total Events': [int(x.sum())],
        })
    
        # ---------- plotting ----------
        class PlotResults:
            pass
    
        out = PlotResults()
        out.eff = p_raw
        out.row_ci_lower = wilson_lo
        out.row_ci_upper = wilson_hi
        out.study_names = subset['Study'].values
        out.n_samplesize = n.astype(int)
        out.n_discordant = x.astype(int)
        out.total_samplesize = int(n.sum())
        out.total_n_discordant = int(x.sum())
        out.mean_effect_re = pooled_pct / 100.0
        out._pooled_ci = (pooled_lo_pct / 100.0, pooled_hi_pct / 100.0)
        out.y = y
        out.v = v
        out.mu_hat_logit = mu_hat  # needed to center funnel plot
    
        return meta_summary_df, out, study_level_df
    
    # ---------------------------------------------------------------------------
    # Run all analyses
    # ---------------------------------------------------------------------------
    overall_results, overall_results_obj, overall_study_df = perform_meta_analysis(data, "Overall")
    grouped_results, grouped_study_dfs, grouped_results_objs = [], [], {}
    
    for category in data['Category'].unique():
        subset = data[data['Category'] == category]
        result_df, result_obj, study_df = perform_meta_analysis(subset, category)
        grouped_results.append(result_df)
        grouped_study_dfs.append(study_df)
        grouped_results_objs[category] = result_obj

        all_results_df = pd.concat([overall_results] + grouped_results, ignore_index=True)
        all_study_level_df = pd.concat([overall_study_df] + grouped_study_dfs, ignore_index=True)
    
        all_results_df.to_excel(
            os.path.join(output_dir, "meta_analysis_summary_overall_and_by_heterogeneity.xlsx"),
            index=False)
        
        all_study_level_df.to_excel(
            os.path.join(output_dir, "meta_analysis_study_level_overall_and_by_heterogeneity.xlsx"),
            index=False)
    
    # ---------------------------------------------------------------------------
    # Forest Plot
    # ---------------------------------------------------------------------------
    def plot_forest_custom(meta_result, category, save=False):
        eff = np.asarray(meta_result.eff)
        ci_lower = np.asarray(meta_result.row_ci_lower)
        ci_upper = np.asarray(meta_result.row_ci_upper)
        study_names = np.asarray(meta_result.study_names)
        n_samples = np.asarray(meta_result.n_samplesize)
        n_events = np.asarray(meta_result.n_discordant)
    
        n = len(eff)
        y_studies = np.arange(n)
        y_pooled = n + 0.5
    
        pooled_effect = meta_result.mean_effect_re
        pooled_lower, pooled_upper = meta_result._pooled_ci
    
        eff_pct = np.clip(eff * 100.0, 0.0, 100.0)
        ci_lower_pct = np.clip(ci_lower * 100.0, 0.0, 100.0)
        ci_upper_pct = np.clip(ci_upper * 100.0, 0.0, 100.0)
        pooled_effect_pct = np.clip(pooled_effect * 100.0, 0.0, 100.0)
        pooled_lower_pct = np.clip(pooled_lower * 100.0, 0.0, 100.0)
        pooled_upper_pct = np.clip(pooled_upper * 100.0, 0.0, 100.0)
        
        # Prevent negative error bars when adjusted CIs cross raw proportions
        lower_error = np.maximum(eff_pct - ci_lower_pct, 0)
        upper_error = np.maximum(ci_upper_pct - eff_pct, 0)
        
        pooled_lower_error = max(pooled_effect_pct - pooled_lower_pct, 0)
        pooled_upper_error = max(pooled_upper_pct - pooled_effect_pct, 0)
            
        def fmt_pct(x):
            if abs(x) < 0.05:
                x = 0.0
            return f"{x:.1f}"
    
        fig_height = (n + 5) * 0.5
        fig, ax = plt.subplots(figsize=(16, fig_height))
        study_name_x, sample_size_x, event_x, event_rate_x = -180, -70, -30, 140
        ax.set_xlim(-180, 160)
    
        header_y = -1
        ax.text(study_name_x, header_y, "Study", ha='left', fontweight='bold', fontsize=16)
        ax.text(sample_size_x, header_y, "Sample size", ha='center', fontweight='bold', fontsize=16)
        ax.text(event_x, header_y, "Event", ha='center', fontweight='bold', fontsize=16)
        ax.text(60, header_y, "Discordance percentage", ha='center', fontweight='bold', fontsize=16)
        ax.text(event_rate_x, header_y, "Event rate (95% CI)", ha='left', fontweight='bold', fontsize=16)
    
        for i in range(n):
            ax.errorbar(
                eff_pct[i], y_studies[i],
                xerr=[[lower_error[i]], [upper_error[i]]],
               # xerr=[[eff_pct[i] - ci_lower_pct[i]], [ci_upper_pct[i] - eff_pct[i]]],
                fmt='o', color='black', capsize=4
            )
            ax.text(study_name_x, y_studies[i], str(study_names[i]), ha='left', va='center', fontsize=16)
            ax.text(sample_size_x, y_studies[i], str(n_samples[i]), ha='center', va='center', fontsize=16)
            ax.text(event_x, y_studies[i], str(n_events[i]), ha='center', va='center', fontsize=16)
            ax.text(
                event_rate_x, y_studies[i],
                f"{fmt_pct(eff_pct[i])}% [{fmt_pct(ci_lower_pct[i])} - {fmt_pct(ci_upper_pct[i])}]",
                ha='left', va='center', fontsize=16
            )
    
        # Pooled line
        ax.errorbar(
            pooled_effect_pct, y_pooled,
            xerr=[[pooled_lower_error], [pooled_upper_error]],
            #xerr=[[pooled_effect_pct - pooled_lower_pct], [pooled_upper_pct - pooled_effect_pct]],
            fmt='s', color='red', capsize=4
        )
        pooled_label = "Pooled percentage\n(random effects)"
        ax.text(study_name_x, y_pooled, pooled_label, ha='left', va='center', fontweight='bold', fontsize=16)
        ax.text(
            sample_size_x, y_pooled, str(meta_result.total_samplesize),
            ha='center', va='center', fontweight='bold', fontsize=16
        )
        ax.text(
            event_rate_x, y_pooled,
            f"{fmt_pct(pooled_effect_pct)}% [{fmt_pct(pooled_lower_pct)} - {fmt_pct(pooled_upper_pct)}]",
            ha='left', va='center', fontweight='bold', fontsize=16
        )
    
        ax.set_ylim(-1.5, n + 1.5)
        ax.invert_yaxis()
        ax.set_xticks([0, 20, 40, 60, 80, 100])
        ax.set_xlabel("Discordance percentage (%)", fontsize=16)
    
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(left=False, labelleft=False, bottom=True)
        plt.tight_layout()
    
        if save:
            plt.savefig(os.path.join(output_dir, f"forest_{category}.png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(output_dir, f"forest_{category}.svg"), dpi=300, bbox_inches="tight")
        plt.show()
    
    # ---------------------------------------------------------------------------
    # Funnel Plot (logit scale, centered at pooled logit)
    # ---------------------------------------------------------------------------
    def plot_funnel(meta_result, category, save=False):
        """
        Funnel on the logit scale: study effects (y) vs SE, centered at pooled logit.
        Draws 95% funnel bounds (mu ± 1.96*SE).
        """
        y = np.asarray(meta_result.y)                 # logit effects
        se = np.sqrt(np.asarray(meta_result.v))      # SE (logit)
        mu = float(getattr(meta_result, "mu_hat_logit", np.nan))
    
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(y, se, alpha=0.7)
    
        # 95% 'funnel' triangle
        if np.isfinite(mu) and np.isfinite(se).any():
            max_se = np.nanmax(se)
            grid = np.linspace(0.0, max_se, 200)
            ax.plot(mu - 1.96*grid, grid, 'k--', linewidth=1)
            ax.plot(mu + 1.96*grid, grid, 'k--', linewidth=1)
            ax.axvline(mu, color='gray', linestyle='solid', linewidth=1)
    
        ax.invert_yaxis()  # conventional funnel
        ax.set_xlabel("Effect (logit proportion)", fontsize=12, fontweight='bold')
        ax.set_ylabel("Standard Error (logit)", fontsize=12, fontweight='bold')
    
        fig.suptitle(f'Funnel Plot — {category}', fontsize=14, fontweight='bold', y=1.02)
    
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
        plt.tight_layout()
        if save:
            plt.savefig(os.path.join(output_dir, f"funnel_{category}.png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(output_dir, f"funnel_{category}.svg"), dpi=300, bbox_inches="tight")
        plt.show()
    
    # ---------------------------------------------------------------------------
    # Run and Save Figures
    # ---------------------------------------------------------------------------
    # Overall
    plot_forest_custom(overall_results_obj, "Overall", save=True)
    plot_funnel(overall_results_obj, "Overall", save=True)
    
    # Each category
    for category, result_obj in grouped_results_objs.items():
        plot_forest_custom(result_obj, category, save=True)
        plot_funnel(result_obj, category, save=True)

pooled_results = pooled_disc_and_forest_funnel_plots()

## Stratified discordance analysis

In [ ]:
"""
Harmonized random-effects meta-analysis of proportions:
  - Study-level and pooled estimates computed on the logit scale; a 0.5 continuity correction is applied only to proportions of 0 or 1
  - Random-effects pooling via Paule–Mandel (method_re="iterated")
  - Hartung–Knapp–Sidik–Jonkman (HKSJ) CIs for pooled random-effects estimates
  - Wilson CIs exported for reference (not used for pooling)
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.meta_analysis import combine_effects
from statsmodels.api import OLS, add_constant
from statsmodels.stats.proportion import proportion_confint
from scipy import stats
from scipy.special import logit, expit
from IPython.display import display
from scipy.stats import norm
from scipy.stats import chi2
import itertools

def run_pooled_disc_and_stratifications():

    sns.set_theme(style="white")
    
    output_dir = r"PATH\FOLDER"
    os.makedirs(output_dir, exist_ok=True)
    
    data = gastric_only.copy()
    
    data['n_samplesize'] = data['combined_sample_size_used_for_discordance'].astype(int)
    data['Category'] = data['Heterogeneity Type']          # expected: intra, inter, intra-inter
    data['n_discordant'] = data['Combined_col_intra_inter discordance rate(n)'].astype(int)
    # Optional: restrict to gastric only
    data = data[data["Gastric_Cohort"] == 1].copy()

    # ---------------------------------------------------------------------------
    # Meta-analysis function (logit + Wilson optional export)
    # ---------------------------------------------------------------------------
    def perform_meta_analysis(subset, category):
    
        # Counts
        x = subset['n_discordant'].to_numpy(dtype=float)
        n = subset['n_samplesize'].to_numpy(dtype=float) 
    
        # ---------- Study-level point estimates ----------
        p_raw = x / n # the discordance rate, but not % yet.
    
        # ---------- Wilson CIs on the raw proportion x/n ----------
        wilson_lo, wilson_hi = proportion_confint(
            count=x.astype(int), nobs=n.astype(int), alpha=0.05, method="wilson" # alpha sets CI to 95%
        )
        wilson_lo = np.clip(wilson_lo, 0, 1) # lower bound: clipped between 0 and 1
        wilson_hi = np.clip(wilson_hi, 0, 1) # higher bound: clipped between 0 and 1
    
        # ---------- Logit transform with boundary-only continuity correction ----------
        # Add 0.5 to the event and non-event counts only when x = 0 or x = n,
        # because the uncorrected logit and its sampling variance are then undefined.
        if np.any(n <= 0):
            raise ValueError("All sample sizes must be positive.")
        if np.any((x < 0) | (x > n)):
            raise ValueError("Discordant counts must satisfy 0 <= x <= n.")

        boundary_mask = (x == 0) | (x == n)
        x_adj = x.copy()
        n_adj = n.copy()
        x_adj[boundary_mask] += 0.5
        n_adj[boundary_mask] += 1.0
        p_adj = x_adj / n_adj
    
        y = logit(p_adj) # 
        v = 1.0/x_adj + 1.0/(n_adj - x_adj) 
        se = np.sqrt(v) 
    
        # ---------- Logit-based study-level CIs ---------- (displays)
        ci_lo_logit = expit(y - 1.96 * se) 
        ci_hi_logit = expit(y + 1.96 * se)
    
        # ---------- Random-effects pooling on logit scale proportions ----------

            # Paule–Mandel estimator for tau² + HKSJ CIs
        res = combine_effects(y, v, method_re="iterated", use_t=True)
        use_t_flag = True
    
        mu_hat = res.mean_effect_re  # pooled *logit* ## calling for pooled effect (logit scale)
    
        # conf_int returns:
        # 0: FE (scale=1), 1: RE (scale=1), 2: FE + HKSJ, 3: RE + HKSJ
        ci_fe, ci_re, ci_fe_hksj, ci_re_hksj = res.conf_int(alpha=0.05, use_t=True)
        ci_re_lo, ci_re_hi = ci_re_hksj  # random-effects + HKSJ-adjustment. 
    
        '''
        Random-effects model: Each study may estimate a different true effect. 
        Effects vary across studies due to both sampling error and heterogeneity.
        --> allows for real differences between studies and is more realistic when heterogeneity exists.
        Use with the KJSH adjustment, because it adjusts the CI for small-study bias and extra uncertainty
    
        Fixed-effects model: All studies share one true effect. 
        Differences in results are due only to sampling error.
        
        '''
    
        # back-transform pooled to %
        pooled_pct    = float(np.clip(expit(mu_hat)   * 100.0, 0, 100)) 
        pooled_lo_pct = float(np.clip(expit(ci_re_lo) * 100.0, 0, 100))
        pooled_hi_pct = float(np.clip(expit(ci_re_hi) * 100.0, 0, 100))
    
        # ---------- Heterogeneity ----------Cochran’s Q test. 
        hom = res.test_homogeneity()
        q, df, p_het = hom.statistic, hom.df, hom.pvalue
        tau2 = res.tau2
        I2 = max(0, (q - df) / q) * 100.0 if q > 0 else 0.0 # 
    
        # ---------- Prediction interval (logit scale) ----------
        
        k = len(y) 
        df_pred = max(k - 2, 1)  
    
        # HKSJ variance for the pooled RE effect:
        se_pooled_hksj = np.sqrt(res.var_hksj_re) 
    
        # Prediction SE includes between-study variance:
        se_pred = np.sqrt(se_pooled_hksj**2 + tau2) 
        t_crit = stats.t.isf(0.025, df_pred) 
    
        pi_lo_logit = mu_hat - t_crit * se_pred 
        pi_hi_logit = mu_hat + t_crit * se_pred
    
        pi_lo_pct = float(np.clip(expit(pi_lo_logit) * 100.0, 0, 100)) 
        pi_hi_pct = float(np.clip(expit(pi_hi_logit) * 100.0, 0, 100))
    
        # ---------- Egger’s regression test (logit scale) ---------- Publication Bias. 
        z = y / se 
        X = add_constant(1.0 / se) 
        egger_model = OLS(z, X).fit() 
        egger_p = egger_model.pvalues[0] 
    
        # ---------- DataFrames for export ---------- NOT POOLED --- just individual study effect-sizes and CIs.
        study_level_df = pd.DataFrame({
            'Category': category,
            'Study': subset['Study'].values,
            'Sample Size': n.astype(int),
            'Events': x.astype(int),
            'Effect Size (proportion)': p_raw,
            'CI Lower (Logit 95%)': ci_lo_logit,    # of the logit transformed proportion values
            'CI Upper (Logit 95%)': ci_hi_logit,    # of the logit transformed proportion values
            'CI Lower (Wilson 95%)': wilson_lo,     # of the raw proportion values
            'CI Upper (Wilson 95%)': wilson_hi,     # of the raw proportion values
            'Effect (logit)': y,
            'SE (logit)': se
        })
    
        meta_summary_df = pd.DataFrame({
            'Category': [category],
            'Pooled % (Random Effects, logit-normal)': [pooled_pct],
            'CI Lower % (95%)': [pooled_lo_pct],
            'CI Upper % (95%)': [pooled_hi_pct],
            'Prediction Interval Lower % (95%)': [pi_lo_pct],
            'Prediction Interval Upper % (95%)': [pi_hi_pct],
            'Between-Study Variance (tau²)': [tau2],
            'Q-Statistic': [q],
            'df': [df],
            'p-value for Heterogeneity': [p_het],
            'I² (%)': [I2],
            'Knapp–Hartung used': [use_t_flag],
            'Egger intercept p-value (logit)': [egger_p],
            'Total Sample Size': [int(n.sum())],
            'Total Events': [int(x.sum())],
        })
    
        # ---------- Attach for plotting ----------
        class PlotResults:
            pass
    
        out = PlotResults()
        out.eff = p_raw
        out.row_ci_lower = wilson_lo
        out.row_ci_upper = wilson_hi
        out.study_names = subset['Study'].values
        out.n_samplesize = n.astype(int)
        out.n_discordant = x.astype(int)
        out.total_samplesize = int(n.sum())
        out.total_n_discordant = int(x.sum())
        out.mean_effect_re = pooled_pct / 100.0
        out._pooled_ci = (pooled_lo_pct / 100.0, pooled_hi_pct / 100.0)
        out.y = y
        out.v = v
        out.mu_hat_logit = mu_hat
        out.method_label = "RE + HKSJ"
        out.q = q
        out.df = df
        out.tau2 = tau2
        out.p_het = p_het
    
        return meta_summary_df, out, study_level_df
    
    # ---------------------------------------------------------------------------
    # Run all analyses
    # ---------------------------------------------------------------------------
    overall_results, overall_results_obj, overall_study_df = perform_meta_analysis(data, "Overall")
    
    grouped_results, grouped_study_dfs, grouped_results_objs = [], [], {}
    for category in data['Category'].unique():
        subset = data[data['Category'] == category]
        result_df, result_obj, study_df = perform_meta_analysis(subset, category)
        grouped_results.append(result_df)
        grouped_study_dfs.append(study_df)
        grouped_results_objs[category] = result_obj
    
    all_results_df = pd.concat([overall_results] + grouped_results, ignore_index=True)
    all_study_level_df = pd.concat([overall_study_df] + grouped_study_dfs, ignore_index=True)
    
    # Save tables
    all_results_df.to_excel(os.path.join(output_dir, f"meta_analysis_summary.xlsx"), index=False)
    display(all_results_df)
    
    # ---------------------------------------------------------------------------
    # Forest Plot
    # ---------------------------------------------------------------------------
    def plot_forest_custom(meta_result, category, save=False):
        eff = np.asarray(meta_result.eff)
        ci_lower = np.asarray(meta_result.row_ci_lower)
        ci_upper = np.asarray(meta_result.row_ci_upper)
        study_names = np.asarray(meta_result.study_names)
        n_samples = np.asarray(meta_result.n_samplesize)
        n_events = np.asarray(meta_result.n_discordant)
    
        n = len(eff)
        y_studies = np.arange(n)
        y_pooled = n + 0.5
    
        pooled_effect = meta_result.mean_effect_re
        pooled_lower, pooled_upper = meta_result._pooled_ci
    
        eff_pct = np.clip(eff * 100.0, 0.0, 100.0)
        ci_lower_pct = np.clip(ci_lower * 100.0, 0.0, 100.0)
        ci_upper_pct = np.clip(ci_upper * 100.0, 0.0, 100.0)
        pooled_effect_pct = np.clip(pooled_effect * 100.0, 0.0, 100.0)
        pooled_lower_pct = np.clip(pooled_lower * 100.0, 0.0, 100.0)
        pooled_upper_pct = np.clip(pooled_upper * 100.0, 0.0, 100.0)

        # Prevent negative error bars when adjusted CIs cross raw proportions
        lower_error = np.maximum(eff_pct - ci_lower_pct, 0)
        upper_error = np.maximum(ci_upper_pct - eff_pct, 0)
        
        pooled_lower_error = max(pooled_effect_pct - pooled_lower_pct, 0)
        pooled_upper_error = max(pooled_upper_pct - pooled_effect_pct, 0)
    
        def fmt_pct(x):
            if abs(x) < 0.05:
                x = 0.0
            return f"{x:.1f}"
    
        fig_height = (n + 5) * 0.5
        fig, ax = plt.subplots(figsize=(16, fig_height))
        study_name_x, sample_size_x, event_x, event_rate_x = -180, -70, -30, 140
        ax.set_xlim(-180, 160)
    
        header_y = -1
        ax.text(study_name_x, header_y, "Study", ha='left', fontweight='bold', fontsize=16)
        ax.text(sample_size_x, header_y, "Sample size", ha='center', fontweight='bold', fontsize=16)
        ax.text(event_x, header_y, "Event", ha='center', fontweight='bold', fontsize=16)
        ax.text(60, header_y, "Discordance percentage", ha='center', fontweight='bold', fontsize=16)
        ax.text(event_rate_x, header_y, "Event rate (95% CI)", ha='left', fontweight='bold', fontsize=16)
    
        for i in range(n):
            ax.errorbar(
                eff_pct[i], y_studies[i],
                #xerr=[[eff_pct[i] - ci_lower_pct[i]], [ci_upper_pct[i] - eff_pct[i]]],
                xerr=[[lower_error[i]], [upper_error[i]]],
                fmt='o', color='black', capsize=4
            )
            ax.text(study_name_x, y_studies[i], str(study_names[i]), ha='left', va='center', fontsize=16)
            ax.text(sample_size_x, y_studies[i], str(n_samples[i]), ha='center', va='center', fontsize=16)
            ax.text(event_x, y_studies[i], str(n_events[i]), ha='center', va='center', fontsize=16)
            ax.text(
                event_rate_x, y_studies[i],
                f"{fmt_pct(eff_pct[i])}% [{fmt_pct(ci_lower_pct[i])} - {fmt_pct(ci_upper_pct[i])}]",
                ha='left', va='center', fontsize=16
            )
    
        # Pooled line
        ax.errorbar(
            pooled_effect_pct, y_pooled,
            #xerr=[[pooled_effect_pct - pooled_lower_pct], [pooled_upper_pct - pooled_effect_pct]],
            xerr=[[pooled_lower_error], [pooled_upper_error]],
            fmt='s', color='red', capsize=4
        )
        pooled_label = "Pooled percentage\n(random effects)"
        ax.text(study_name_x, y_pooled, pooled_label, ha='left', va='center', fontweight='bold', fontsize=16)
        ax.text(
            sample_size_x, y_pooled, str(meta_result.total_samplesize),
            ha='center', va='center', fontweight='bold', fontsize=16
        )
        ax.text(
            event_rate_x, y_pooled,
            f"{fmt_pct(pooled_effect_pct)}% [{fmt_pct(pooled_lower_pct)} - {fmt_pct(pooled_upper_pct)}]",
            ha='left', va='center', fontweight='bold', fontsize=16
        )
    
        ax.set_ylim(-1.5, n + 1.5)
        ax.invert_yaxis()
        ax.set_xticks([0, 20, 40, 60, 80, 100])
        ax.set_xlabel("Discordance percentage (%)", fontsize=16)
    
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(left=False, labelleft=False, bottom=True)
        plt.tight_layout()
    
        if save:
            plt.savefig(os.path.join(output_dir, f"forest_{category}.png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(output_dir, f"forest_{category}.svg"), dpi=300, bbox_inches="tight")
        plt.show()
    
    # ---------------------------------------------------------------------------
    # Funnel Plot (logit scale, centered at pooled logit)
    # ---------------------------------------------------------------------------
    def plot_funnel(meta_result, category, save=False):
        """
        Funnel on the logit scale: study effects (y) vs SE, centered at pooled logit.
        Draws 95% funnel bounds (mu ± 1.96*SE).
        """
        y = np.asarray(meta_result.y)                 # logit effects
        se = np.sqrt(np.asarray(meta_result.v))       # SE (logit)
        mu = float(getattr(meta_result, "mu_hat_logit", np.nan))
    
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(y, se, alpha=0.7)
    
        # 95% 'funnel' triangle
        if np.isfinite(mu) and np.isfinite(se).any():
            max_se = np.nanmax(se)
            grid = np.linspace(0.0, max_se, 200)
            ax.plot(mu - 1.96*grid, grid, 'k--', linewidth=1)
            ax.plot(mu + 1.96*grid, grid, 'k--', linewidth=1)
            ax.axvline(mu, color='gray', linestyle='solid', linewidth=1)
    
        ax.invert_yaxis()  # conventional funnel
        ax.set_xlabel("Effect (logit proportion)", fontsize=12, fontweight='bold')
        ax.set_ylabel("Standard Error (logit)", fontsize=12, fontweight='bold')
        fig.suptitle(f'Funnel Plot — {category}', fontsize=14, fontweight='bold', y=1.02)
    
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
        plt.tight_layout()
        if save:
            plt.savefig(os.path.join(output_dir, f"funnel_{category}.png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(output_dir, f"funnel_{category}.svg"), dpi=300, bbox_inches="tight")
        plt.show()
    
    # ---------------------------------------------------------------------------
    # Run and Save Figures
    # ---------------------------------------------------------------------------
    # Overall
    
    plot_forest_custom(overall_results_obj, "Overall", save=True)
    plot_funnel(overall_results_obj, "Overall", save=True)
    
    # Each category
    for category, result_obj in grouped_results_objs.items():
        plot_forest_custom(result_obj, category, save=True)
        plot_funnel(result_obj, category, save=True)
    
    # ###############################################################################
    # #             Additional Subgroup Meta‐Analyses (Stratified)              #
    # ###############################################################################
    
    all_summary_dfs = []
    all_results_objs = {}
    
    for category, category_data in data.groupby("Heterogeneity Type"):
    
        # Optional: add unstratified pooled result for this heterogeneity type
        summary_df, res_obj, study_df = perform_meta_analysis(
            category_data,
            f"{category} | Overall"
        )
    
        all_summary_dfs.append(summary_df)
        all_results_objs[f"{category} | Overall"] = res_obj
    
        # Stratifications within this heterogeneity type
        for col, info in STRATIFICATIONS.items():
    
            if col not in category_data.columns:
                print(f"{category} - {col}: column missing, skipped")
                continue
    
            mapping = info["mapping"]
            mode = info["mode"]
    
            if mode == "both":
                for val, label in mapping.items():
                    subset = category_data[category_data[col] == val]
    
                    if not subset.empty:
                        group_label = f"{category} | {col}: {label}"
    
                        summary_df, res_obj, study_df = perform_meta_analysis(
                            subset,
                            group_label
                        )
    
                        all_summary_dfs.append(summary_df)
                        all_results_objs[group_label] = res_obj
    
            elif mode == "only1":
                subset = category_data[category_data[col] == 1]
    
                if not subset.empty:
                    group_label = f"{category} | {col}: {mapping.get(1, '1')}"
    
                    summary_df, res_obj, study_df = perform_meta_analysis(
                        subset,
                        group_label
                    )
    
                    all_summary_dfs.append(summary_df)
                    all_results_objs[group_label] = res_obj

    ###############################################################################
    #                Combine and Save Summary Results                          #
    ###############################################################################
    all_results_df = pd.concat(all_summary_dfs, ignore_index=True)

    all_results_df.to_excel(os.path.join(output_dir, f"meta_analysis_summary_stratifications.xlsx"), index=False)
    
    ###############################################################################
    #                Pairwise Statistical Comparisons DataFrame                #
    ###############################################################################

    def get_pooled_logit_and_se(result_obj):
        res = combine_effects(
            result_obj.y,
            result_obj.v,
            method_re="iterated",
            use_t=True
        )
        mu_logit = float(res.mean_effect_re)
        se_logit = float(res.sd_eff_w_re)
        return mu_logit, se_logit

    comparison_rows = []
 
    strat_groups = {}

    for key in all_results_objs.keys():
        if key == "Overall":
            continue
        if ":" in key:
            strat_var = key.split(":")[0].strip()
            strat_groups.setdefault(strat_var, []).append(key)
 
    for strat_var, keys in strat_groups.items():
        if len(keys) > 1:
            for key1, key2 in itertools.combinations(keys, 2):
                logit1, se1 = get_pooled_logit_and_se(all_results_objs[key1])
                logit2, se2 = get_pooled_logit_and_se(all_results_objs[key2])
                diff = logit1 - logit2
                se_diff = np.sqrt(se1**2 + se2**2)
                z_stat = diff / se_diff if se_diff > 0 else np.nan
                p_value = 2 * (1 - norm.cdf(abs(z_stat))) if se_diff > 0 else np.nan
 
                rate1 = float(all_results_objs[key1].mean_effect_re)
                rate2 = float(all_results_objs[key2].mean_effect_re)
                
                category1, label1 = key1.split(": ", 1)
                category2, label2 = key2.split(": ", 1)

                heterogeneity_type = category1.split(" | ", 1)[0]

 
                comparison_rows.append({
                    "Heterogeneity Type": heterogeneity_type,
                    "Comparison": f"{label1} vs {label2}",
                    "Discordance rate 1": f"{rate1:.3f}",
                    "Discordance rate 2": f"{rate2:.3f}",
                    "Statistic": f"{z_stat:.3f}" if np.isfinite(z_stat) else "NA",
                    "p-value (z-test)": f"{p_value:.3f}" if np.isfinite(z_stat) else "NA",
                    "Method (RE)": all_results_objs[key1].method_label
                })
 
    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df)
    comparison_df.to_excel(os.path.join(output_dir, f"meta_analysis_pairwise_COMPARISONS.xlsx"), index=False)

    return {
        "overall_obj": overall_results_obj,      # pooled overall
        "all_results_objs": all_results_objs,    # dict: "Overall", "Asian_Cohort: Asian", ...
        "all_results_df": all_results_df,        # summary table
        "comparison_df": comparison_df           # pairwise comparisons table
    }

pooled_strati = run_pooled_disc_and_stratifications()

all_results_objs = pooled_strati["all_results_objs"]
overall_results_obj = pooled_strati["overall_obj"]

## Sensitivity analysis

In [ ]:
###############################################################################
#                   CLDN-18.2 Discordance Sensitivity Analysis                #
###############################################################################

import os
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.stats import chi2
from scipy import stats
from statsmodels.stats.meta_analysis import combine_effects
from IPython.display import display


def run_sensitivity_func():

    ###############################################################################
    #  Setup and Data Reading
    ###############################################################################

    sns.set_theme(style="white")

    output_dir = r"PATH/FOLDER"
    os.makedirs(output_dir, exist_ok=True)

    data = gastric_only.copy()

    # Basic columns
    data["Category"] = data["Heterogeneity Type"]

    data["n_samplesize"] = pd.to_numeric(
        data["combined_sample_size_used_for_discordance"],
        errors="coerce"
    )

    data["n_discordant"] = pd.to_numeric(
        data["Combined_col_intra_inter discordance rate(n)"],
        errors="coerce"
    )

    data = data.dropna(subset=["Category", "n_samplesize", "n_discordant"]).copy()
    data["n_samplesize"] = data["n_samplesize"].astype(int)
    data["n_discordant"] = data["n_discordant"].astype(int)
    data["Percentage"] = data["n_discordant"] / data["n_samplesize"]

    ###############################################################################
    #  Harmonized Meta-Analysis Function
    ###############################################################################

    def perform_meta_analysis(subset, group_label, alpha=0.05):

        x = subset["n_discordant"].to_numpy(dtype=float)
        n = subset["n_samplesize"].to_numpy(dtype=float)

        # ---------- Logit transform with boundary-only continuity correction ----------
        # Add 0.5 to the event and non-event counts only when x = 0 or x = n,
        # because the uncorrected logit and its sampling variance are then undefined.
        if np.any(n <= 0):
            raise ValueError("All sample sizes must be positive.")
        if np.any((x < 0) | (x > n)):
            raise ValueError("Discordant counts must satisfy 0 <= x <= n.")

        boundary_mask = (x == 0) | (x == n)
        x_adj = x.copy()
        n_adj = n.copy()
        x_adj[boundary_mask] += 0.5
        n_adj[boundary_mask] += 1.0
        p_adj = x_adj / n_adj

        y = np.log(p_adj / (1.0 - p_adj))
        v = 1.0 / x_adj + 1.0 / (n_adj - x_adj)

        use_kh = False
        try:
            res = combine_effects(y, v, method_re="iterated", use_t=True)
            use_kh = True
        except TypeError:
            res = combine_effects(y, v, method_re="iterated")
            use_kh = False

        mu_hat = res.mean_effect_re

        if use_kh:
            ci_fe, ci_re, ci_fe_hksj, ci_re_hksj = res.conf_int(alpha=alpha, use_t=True)
            ci_lo_logit, ci_hi_logit = ci_re_hksj
        else:
            ci_lo_logit, ci_hi_logit = res.conf_int(alpha=alpha)[1]

        inv_logit = lambda z: 1.0 / (1.0 + np.exp(-z))

        pooled_prop = inv_logit(mu_hat)
        pooled_lo_prop = inv_logit(ci_lo_logit)
        pooled_hi_prop = inv_logit(ci_hi_logit)

        hom = res.test_homogeneity()
        q_stat = hom.statistic
        df_q = hom.df
        pvalue_q = hom.pvalue
        tau2 = res.tau2

        I2 = (max(0.0, (q_stat - df_q)) / q_stat * 100.0) if q_stat > 0 else 0.0

        # I2 95% CI
        i2_lower = 0.0
        i2_upper = 0.0

        if df_q > 0 and q_stat > 0:
            chiQ_L = chi2.ppf(alpha / 2.0, df_q)
            chiQ_U = chi2.ppf(1.0 - alpha / 2.0, df_q)

            H2_low = q_stat / chiQ_U
            H2_high = q_stat / chiQ_L

            def H2_to_I2(h2):
                if h2 <= 1.0:
                    return 0.0
                return (h2 - 1.0) / h2

            i2_lower = H2_to_I2(H2_low) * 100
            i2_upper = min(1.0, H2_to_I2(H2_high)) * 100

        # Prediction interval
        k = len(y)
        df_pred = max(k - 2, 1)

        if use_kh and hasattr(res, "var_hksj_re"):
            var_mu = float(res.var_hksj_re)
        else:
            var_mu = float(getattr(res, "var_re", 0.0))

        se_pred = np.sqrt(var_mu + tau2)
        t_crit = stats.t.isf(0.025, df_pred)

        pi_lo_logit = mu_hat - t_crit * se_pred
        pi_hi_logit = mu_hat + t_crit * se_pred

        pi_lo_prop = inv_logit(pi_lo_logit)
        pi_hi_prop = inv_logit(pi_hi_logit)

        pooled_effect_str = f"{pooled_prop:.3f} ({pooled_lo_prop:.3f}-{pooled_hi_prop:.3f})"
        i2_str = f"{I2:.1f}% ({i2_lower:.1f}%-{i2_upper:.1f}%)"
        method_label = "Paule-Mandel RE + HKSJ" if use_kh else "Paule-Mandel RE (no HKSJ)"

        meta_summary_df = pd.DataFrame({
            "Group": [group_label],
            "Pooled Effect Size (95% CI)": [pooled_effect_str],
            "Between-Study Variance (tau²)": [tau2],
            "Q-Statistic": [q_stat],
            "p-value for Heterogeneity": [pvalue_q],
            "I² (95% CI)": [i2_str],
            "Number of Studies": [subset.shape[0]],
            "Pooled Patients": [int(subset["n_samplesize"].sum())],
            "Method": [method_label],
            "Prediction Interval Lower (prop)": [pi_lo_prop],
            "Prediction Interval Upper (prop)": [pi_hi_prop],
            "Prediction Interval (95%)": [f"{pi_lo_prop:.3f}-{pi_hi_prop:.3f}"]
        })

        res.mean_effect_re_prop = pooled_prop
        res.ci_prop = (pooled_lo_prop, pooled_hi_prop)
        res.method_label = method_label
        res.pi_prop = (pi_lo_prop, pi_hi_prop)

        return meta_summary_df, res

    ###############################################################################
    #  Extended Sensitivity Analysis
    ###############################################################################

    def perform_extended_sensitivity_analysis(scope_data, scope_label):

        overall_summary, overall_res = perform_meta_analysis(
            scope_data,
            f"{scope_label} | All Studies"
        )

        overall_pooled = overall_res.mean_effect_re_prop
        overall_lo, overall_hi = overall_res.ci_prop

        flagged_summaries = []
        flagged_studies = []

        for idx, row in scope_data.iterrows():
            if scope_data.shape[0] <= 2:
                continue

            subset = scope_data.drop(idx)

            leave_summary, leave_res = perform_meta_analysis(
                subset,
                f"{scope_label} | Excluding: {row['Study']}"
            )

            leave_pooled = leave_res.mean_effect_re_prop

            if (leave_pooled < overall_lo) or (leave_pooled > overall_hi):
                flagged_studies.append(row["Study"])
                flagged_summaries.append(leave_summary)

        if flagged_studies:
            data_sensitivity = scope_data[~scope_data["Study"].isin(flagged_studies)]
            combined_label = f"{scope_label} | Excluding all flagged: " + ", ".join(flagged_studies)
            combined_summary, _ = perform_meta_analysis(data_sensitivity, combined_label)
        else:
            combined_summary = pd.DataFrame()

        summary_rows = [overall_summary]

        if flagged_summaries:
            summary_rows.extend(flagged_summaries)

        if not combined_summary.empty:
            summary_rows.append(combined_summary)

        summary_table = pd.concat(summary_rows, ignore_index=True)

        return summary_table, flagged_studies

    ###############################################################################
    #  Run Sensitivity Analysis: Overall + Per Heterogeneity Type
    ###############################################################################

    all_summary_tables = []
    all_flagged_tables = []

    scope_datasets = {"Overall": data}

    for hetero_type, subset in data.groupby("Heterogeneity Type"):
        scope_datasets[hetero_type] = subset

    for scope_label, scope_data in scope_datasets.items():

        summary_table, flagged = perform_extended_sensitivity_analysis(
            scope_data,
            scope_label
        )

        summary_table.insert(0, "Scope", scope_label)
        all_summary_tables.append(summary_table)

        print(f"\n=== Sensitivity Analysis Summary: {scope_label} ===")
        display(summary_table)

        print(f"\n=== Flagged Studies: {scope_label} ===")
        if flagged:
            for f in flagged:
                print(" -", f)

            flagged_df = pd.DataFrame({
                "Scope": scope_label,
                "Flagged Studies": flagged
            })
            all_flagged_tables.append(flagged_df)

        else:
            print("No influential studies flagged.")

    final_summary_table = pd.concat(all_summary_tables, ignore_index=True)

    summary_path = os.path.join(
        output_dir,
        "meta_analysis_extended_sensitivity_summary_overall_and_by_heterogeneity.xlsx"
    )

    final_summary_table.to_excel(summary_path, index=False)

    if all_flagged_tables:
        final_flagged_table = pd.concat(all_flagged_tables, ignore_index=True)
    else:
        final_flagged_table = pd.DataFrame(columns=["Scope", "Flagged Studies"])

    flagged_path = os.path.join(
        output_dir,
        "meta_analysis_flagged_studies_overall_and_by_heterogeneity.xlsx"
    )

    final_flagged_table.to_excel(flagged_path, index=False)

    print(f"\nSaved sensitivity summary to:\n  {summary_path}")
    print(f"\nSaved flagged studies list to:\n  {flagged_path}")

    return {
        "summary_table": final_summary_table,
        "flagged_table": final_flagged_table
    }

sensitivity_results = run_sensitivity_func()
sensitivity_summary = sensitivity_results["summary_table"]
sensitivity_flagged = sensitivity_results["flagged_table"]

## Individual Forest plots for intra- and inter- tumoural heterogeneity stratifications

In [ ]:
###############################################################################
#                     Subgroup Forest Plots (Meta-Analysis)                   #
#                  Consistent with Iterated RE + KH (Logit)                   #
###############################################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base_output_dir = r"PATH/FOLDER"
os.makedirs(base_output_dir, exist_ok=True)

data = gastric_only.copy() 

# Ensure consistent naming
data['n_samplesize'] = data['combined_sample_size_used_for_discordance'].astype(int)
data['n_discordant'] = data['Combined_col_intra_inter discordance rate(n)'].astype(int)

# =============================================================================
# Function to Generate Subgroup Forest Plot
# =============================================================================

def safe_filename(text):
    return (
        str(text)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace("|", "_")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )

def plot_forest_subgroups(
    meta_results_dict,
    category,
    include_overall=False,
    overall_obj=None,
    save=False,
    base_output_dir=base_output_dir,
    source_data=data
):
    """
    Generate subgroup-level forest plots using pooled meta-analysis results.

      - Uses pooled estimates on proportion scale (.mean_effect_re_prop / .ci_prop).
      - If missing, assumes logit scale and back-transforms (inv_logit).
      - Enforces Iterated RE (+KH) consistency across all results.
      - Falls back to raw reconstruction for N and Events if totals missing.
      
    """
    
    for lbl, res in meta_results_dict.items():
        ml = getattr(res, "method_label", "")
        if "HKSJ" not in ml:
            raise RuntimeError(f"[Error] {lbl} unexpected method_label='{ml}'")

    # -------------------------------------------------------------------------
    #  Helper Functions
    # -------------------------------------------------------------------------
    def _inv_logit(z):
        return 1.0 / (1.0 + np.exp(-z))

    def _pooled_prop_and_ci(res, alpha=0.05):
        pe = float(getattr(res, "mean_effect_re", np.nan))
        lo, hi = getattr(res, "_pooled_ci", (np.nan, np.nan))
        return pe, float(lo), float(hi)

        # Fallback (logit back-transform)
        pe_logit = getattr(res, "mean_effect_re", np.nan)
        try:
            lo_logit, hi_logit = res.conf_int(alpha=alpha)[1]
        except Exception:
            lo_logit, hi_logit = np.nan, np.nan
        if not np.all(np.isfinite([pe_logit, lo_logit, hi_logit])):
            return np.nan, np.nan, np.nan
        return _inv_logit(pe_logit), _inv_logit(lo_logit), _inv_logit(hi_logit)

    def _attached_totals(res):
        """Return total N and events from the meta-analysis result object."""
        tn = getattr(res, "total_samplesize", np.nan)
        te = getattr(res, "total_n_discordant", np.nan)
        if not np.isfinite(tn):
            vec = getattr(res, "n_samplesize", None)
            if vec is not None:
                tn = int(np.nansum(np.asarray(vec, dtype=float)))
        if not np.isfinite(te):
            vec = getattr(res, "n_discordant", None)
            if vec is not None:
                te = int(np.nansum(np.asarray(vec, dtype=float)))
        return tn, te


    label_code_map = {
    "Asian_Cohort": {"Asian cohorts": 1, "Non-Asian cohorts": 0, "mode": "both"},    
    "Sample_dSize": {"≥ median sample size": 1, "<median sample size": 0, "mode": "both"},
    "Stage_IV": {"Stage IV": 1, "Stage I-III": 0, "mode": "both"},
    "Lymph": {"mapping": {1: "Lymph node", 0: "Non-lymph node"}, "mode": "both"},
    "Peritoneal": {"mapping": {1: "Peritoneal", 0: "Non-peritoneal"}, "mode": "both"},
    "Liver": {"mapping": {1: "Liver", 0: "Non-liver"}, "mode": "both"},
    "Majority_G3": {"mapping": {1: "Majority G3", 0: "Majority <G3"}, "mode": "both"},
    "Technique": {"mapping": {1: "IHC", 0: "IHC & TMA"}, "mode": "both"},
    "assay_cate_new": {"mapping": {1: "RxDx", 0: "LDT"}, "mode": "both"},
    "automation_re-ex": {"mapping": {1: "Automated", 0: "Manual"}, "mode": "both"},
    "Threshold": {"mapping": {1: ">75%", 0: "Other"}, "mode": "both"},
    "Cohort_Naive_b": {"mapping": {1: "Treatment-naïve", 0: "Previously treated"}, "mode": "both"},
}

    def _reconstruct_totals_from_label(lbl):
        """Fallback: derive total N and Events from source_data using label mapping."""
        if ": " not in lbl:
            return np.nan, np.nan
        col, lab = lbl.split(": ", 1)
        if col not in label_code_map:
            return np.nan, np.nan
        code_map = label_code_map[col]
        key = lab.strip()
        if key not in code_map:
            # Normalize label formatting
            alt_map = {k.replace("_", " ").replace("-", " ").strip(): v for k, v in code_map.items()}
            key = key.replace("_", " ").replace("-", " ").strip()
            if key not in alt_map:
                return np.nan, np.nan
            code = alt_map[key]
        else:
            code = code_map[key]

        if col == "Type":
            mask = (source_data["Type"] == code) | (
                "Category" in source_data.columns and (source_data["Category"] == code)
            )
        else:
            if col not in source_data.columns:
                return np.nan, np.nan
            mask = (source_data[col] == code)

        if hasattr(mask, "any") and mask.any():
            tn = int(source_data.loc[mask, "n_samplesize"].sum())
            te = int(source_data.loc[mask, "n_discordant"].sum())
            return tn, te
        return np.nan, np.nan

    # -------------------------------------------------------------------------
    #  Collect Pooled Estimates for Each Subgroup
    # -------------------------------------------------------------------------
    
    items = sorted(meta_results_dict.items(), key=lambda x: x[0].lower())

    group_labels, pe_list, lo_list, hi_list = [], [], [], []
    ns_list, ev_list = [], []

    for label, res in items:
        pe, lo, hi = _pooled_prop_and_ci(res)
        tn, te = _attached_totals(res)
        if not np.isfinite(tn) or not np.isfinite(te):
            tn2, te2 = _reconstruct_totals_from_label(label)
            tn = tn if np.isfinite(tn) else tn2
            te = te if np.isfinite(te) else te2

        group_labels.append(label)
        pe_list.append(pe); lo_list.append(lo); hi_list.append(hi)
        ns_list.append(tn);  ev_list.append(te)

    # -------------------------------------------------------------------------
    #  Sanity: Drop NaNs, Clip to [0,1]
    # -------------------------------------------------------------------------
    mask = np.isfinite(pe_list) & np.isfinite(lo_list) & np.isfinite(hi_list)
    group_labels = [g for g, m in zip(group_labels, mask) if m]
    pe_arr = np.clip(np.array(pe_list, dtype=float)[mask], 0.0, 1.0)
    lo_arr = np.clip(np.array(lo_list, dtype=float)[mask], 0.0, 1.0)
    hi_arr = np.clip(np.array(hi_list, dtype=float)[mask], 0.0, 1.0)
    ns_arr = np.array(ns_list, dtype=float)[mask]
    ev_arr = np.array(ev_list, dtype=float)[mask]

    # Convert to percent scale
    pe_pct = pe_arr * 100.0
    lo_pct = lo_arr * 100.0
    hi_pct = hi_arr * 100.0

    # -------------------------------------------------------------------------
    #  Build Figure
    # -------------------------------------------------------------------------
    n = len(group_labels)
    extra = 1 if include_overall else 0
    fig_height = (n + extra + 5) * 0.5
    fig, ax = plt.subplots(figsize=(16, fig_height))

    # Layout positions
    study_name_x = -150
    sample_size_x = -70
    event_x       = -30
    event_rate_x  = 140
    ax.set_xlim(-150, 160)

    # Header
    header_y = -1
    ax.text(study_name_x,  header_y, "Sub-group", ha='left', va='bottom', fontweight='bold', fontsize=16)
    ax.text(sample_size_x, header_y, "Sample size", ha='center', va='bottom', fontweight='bold', fontsize=16)
    ax.text(event_x,       header_y, "Event", ha='center', va='bottom', fontweight='bold', fontsize=16)
    ax.text(60, header_y, "Discordance percentage", ha='center', va='bottom', fontweight='bold', fontsize=16)
    ax.text(event_rate_x,  header_y, "Pooled proportion (95% CI)", ha='left', va='bottom', fontweight='bold', fontsize=16)

    # Subgroup rows
    y = np.arange(n)
    for i in range(n):
        ax.errorbar(
            pe_pct[i], y[i],
            #xerr=[[pe_pct[i] - lo_pct[i]], [hi_pct[i] - pe_pct[i]]],
            xerr=[[max(pe_pct[i] - lo_pct[i], 0)],[max(hi_pct[i] - pe_pct[i], 0)]],
            fmt='o', color='black', capsize=4
        )
        label = group_labels[i].split(": ", 1)[1] if ": " in group_labels[i] else group_labels[i]
        ns_txt = "—" if not np.isfinite(ns_arr[i]) else str(int(ns_arr[i]))
        ev_txt = "—" if not np.isfinite(ev_arr[i]) else str(int(ev_arr[i]))
        ax.text(study_name_x,  y[i], label, ha='left', va='center', fontsize=16)
        ax.text(sample_size_x, y[i], ns_txt, ha='center', va='center', fontsize=16)
        ax.text(event_x,       y[i], ev_txt, ha='center', va='center', fontsize=16)
        ax.text(
            event_rate_x, y[i],
            f"{pe_pct[i]:.1f}% [{lo_pct[i]:.1f}–{hi_pct[i]:.1f}]",
            ha='left', va='center', fontsize=16
        )

    # Optional pooled overall row
    if include_overall and overall_obj is not None:
        o_pe, o_lo, o_hi = _pooled_prop_and_ci(overall_obj)
        o_ts, _ = _attached_totals(overall_obj)
        y_overall = n + 0.5
        ax.errorbar(o_pe*100.0, y_overall,
                    xerr=[[o_pe*100.0 - o_lo*100.0], [o_hi*100.0 - o_pe*100.0]],
                    fmt='s', color='red', capsize=4)
        ax.text(study_name_x, y_overall, "Pooled percentage\n(random effects)",
                ha='left', va='center', fontweight='bold', fontsize=16)
        ax.text(sample_size_x, y_overall,
                "—" if not np.isfinite(o_ts) else str(int(o_ts)),
                ha='center', va='center', fontweight='bold', fontsize=16)
        ax.text(event_rate_x, y_overall,
                f"{o_pe*100.0:.2f}% [{o_lo*100.0:.2f}–{o_hi*100.0:.2f}]",
                ha='left', va='center', fontweight='bold', fontsize=16)
        ax.set_ylim(-1.5, n + 2)
    else:
        ax.set_ylim(-1.5, n)

    ax.invert_yaxis()
    ax.set_xticks([0, 20, 40, 60, 80, 100])
    ax.set_xlabel("Discordance percentage (%)", fontsize=16)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=False, labelleft=False, bottom=True)
    plt.tight_layout()

    if save:
        filename = safe_filename(category)
        plt.savefig(os.path.join(base_output_dir, f"forest_{filename}.png"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(base_output_dir, f"forest_{filename}.svg"), dpi=300, bbox_inches="tight")

    plt.show()

    # Export summary table for traceability
    out = pd.DataFrame({
        "Label": group_labels,
        "Pooled proportion": pe_arr,
        "CI low": lo_arr,
        "CI high": hi_arr,
        "Total N": ns_arr,
        "Events": ev_arr,
    })
    return out

# =============================================================================
# Build Subgroup Dictionaries and Plot
# =============================================================================
assert "all_results_objs" in globals() and len(all_results_objs) > 0, \
    "Run the stratified meta-analysis first before executing sub-forest plots."

def get_scope_results(scope, stratification):
    prefix = f"{scope} | {stratification}:"
    return {
        k: v
        for k, v in all_results_objs.items()
        if k.startswith(prefix)
    }

plot_specs = {
    "Geography": "Asian_Cohort",
    "Sample size": "GSample_dSize",
    "Stage IV": "Stage_IV",
    "Lymph-node mets" : "Lymph",
    "Peritoneal mets": "Peritoneal",
    "Liver mets" :"Liver",
    "Majority G3": "Majority_G3",
    "Technique": "Technique",
    "Assay category": "assay_cate_new",
    "Automation": "automation_re-ex",
    "Cut-off": "Threshold",
    "Treatment History": "Cohort_Naive_b",
}

for scope in ["Intra-tumoural (T-T)", "Inter-tumoural (T-T)"]:

    scope_output_dir = os.path.join(base_output_dir, safe_filename(scope))
    os.makedirs(scope_output_dir, exist_ok=True)

    for plot_title, strat_col in plot_specs.items():

        subgroup_results = get_scope_results(scope, strat_col)

        if subgroup_results:
            subgroup_df = plot_forest_subgroups(
                subgroup_results,
                category=f"{scope} - {plot_title}",
                save=True,
                base_output_dir=scope_output_dir
            )

            subgroup_df.to_excel(
                os.path.join(
                    scope_output_dir,
                    f"sub_forest_plot_{safe_filename(strat_col)}_summary.xlsx"
                ),
                index=False
            )
        else:
            print(f"No results found for {scope} | {strat_col}")

# Cross-Cancer

In [ ]:
STRATIFICATIONS: dict[str, dict[str, Any]] = {
    "Asian_Cohort": {"mapping": {1: "Asian", 0: "Non-Asian"}, "mode": "both"},
    "Gastric_Cohort": {"mapping": {1: "Gastric", 0: "Non-gastric"}, "mode": "both"},
    "Stage_IV": {"mapping": {1: "Stage IV", 0: "Stage I/II/III"}, "mode": "both"},
    "Sample_dSize": {"mapping": {1: "≥ median sample size", 0: "< median sample size"}, "mode": "both"}, # for ALL papers (NOT gastric ONLY)
    "Majority_G3": {"mapping": {1: "Majority G3", 0: "Majority <G3"}, "mode": "both"},
    "Lymph": {"mapping": {1: "Lymph-node", 0: "Non-lymph node"}, "mode": "both"},
    "Peritoneal": {"mapping": {1: "Peritoneal", 0: "Non-peritoneal"}, "mode": "both"},
    "Liver": {"mapping": {1: "Liver", 0: "Non-liver"}, "mode": "both"},
    "Technique": {"mapping": {1: "IHC", 0: "IHC & TMA"}, "mode": "both"},
    "assay_cate_new": {"mapping": {1: "RxDx", 0: "LDT"}, "mode": "both"},
    "automation_re-ex": {"mapping": {1: "Automated", 0: "Manual"}, "mode": "both"},
    "Investigation_Bias": {"mapping": {1: "Bias; only positive cases checked", 0: "No bias"}, "mode": "both"},
    "Threshold": {"mapping": {1: ">75%", 0: "Other"}, "mode": "both"},
    "Cohort_Naive_b": {"mapping": {1: "Naive", 0: "Not naive"}, "mode": "both"},
}

## Study characteristics 

In [ ]:
results = {}

fig_dir = r"PATH/FOLDER"
os.makedirs(fig_dir, exist_ok=True)

# Overall unique papers
unique_papers = df0["article nr"].nunique()
results["Overall unique nr of papers"] = unique_papers

# Unique paper counts per heterogeneity type
paper_counts = (
    df0
    .groupby("Heterogeneity Type")["article nr"]
    .nunique()
)
results["Paper Split"] = paper_counts

# Overlap: papers that have both intra and inter heterogeneity rows
intra_papers = set(
    df0.loc[
        df0["Heterogeneity Type"] == "Intra-tumoural (T-T)",
        "article nr"
    ].dropna()
)
inter_papers = set(
    df0.loc[
        df0["Heterogeneity Type"] == "Inter-tumoural (T-T)",
        "article nr"
    ].dropna()
)
overlap_papers = intra_papers & inter_papers
results["Papers with both intra and inter"] = len(overlap_papers)
results["Overlapping paper article nrs"] = sorted(overlap_papers)

# Numeric patient col
df0["combined_sample_size_used_for_discordance"] = pd.to_numeric(
    df0["combined_sample_size_used_for_discordance"],
    errors="coerce"
)
patient_nr_col = 'combined_sample_size_used_for_discordance'

# Patient counts: Overall vs split per heterogeneity type
overall_patients = df0[patient_nr_col].sum()
results["Overall patients"] = int(overall_patients)

# Patient counts and distribution per heterogeneity type
patients_split = (
    df0
    .groupby("Heterogeneity Type")[patient_nr_col]
    .agg(
        total="sum",
        median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        min="min",
        max="max",
        n_rows="count"
    )
)
patients_split["IQR"] = patients_split["Q3"] - patients_split["Q1"]
results["Total (g) patients split"] = patients_split

# Cohort collection
def year_range(series):
    years = pd.to_numeric(series, errors="coerce").dropna()
    if years.empty:
        return "NR"
    return f"{int(years.min())} - {int(years.max())}"
    
def cohort_collection_range(series):
    year_ranges = (
        series
        .astype(str)
        .str.extract(r"(?P<start_year>\d{4})\s*[-–]\s*(?P<end_year>\d{4})")
        .apply(pd.to_numeric, errors="coerce")
        .dropna()
    )
    if year_ranges.empty:
        return "NR"
    start_year = int(year_ranges["start_year"].min())
    end_year = int(year_ranges["end_year"].max())
    return f"{start_year} - {end_year}"
    
# --- Publication years: overall ---
results["Publication years range overall"] = year_range(df0["Year"])

# --- Publication years: per heterogeneity type ---
publication_years_split = (
    df0
    .groupby("Heterogeneity Type")["Year"]
    .apply(year_range, include_groups=False)
)
results["Publication years range split"] = publication_years_split

# --- Cohort collection years: overall ---
results["Cohort collection range overall"] = cohort_collection_range(
    df0["Cohort date range"]
)
# --- Cohort collection years: per heterogeneity type ---
cohort_collection_split = (
    df0
    .groupby("Heterogeneity Type")["Cohort date range"]
    .apply(cohort_collection_range, include_groups=False)
)
results["Cohort collection range split"] = cohort_collection_split

# Country distribution
def plot_country_pie(data, label, fig_dir):
    country_counts = data["Country"].value_counts(dropna=True)
    results_key = f"Country counts - {label}"
    results[results_key] = country_counts

    if country_counts.empty:
        print(f"No country data available for {label}")
        return

    ordered_countries = [c for c in country_order if c in country_counts.index]
    other_countries = [c for c in country_counts.index if c not in country_order]
    country_counts = country_counts.loc[ordered_countries + other_countries]

    country_colors = [
        color_map.get(country, "#CCCCCC")
        for country in country_counts.index
    ]

    plt.figure(figsize=(8, 8))
    country_counts.plot.pie(
        autopct="%1.1f%%",
        colors=country_colors,
        startangle=90,
        wedgeprops={"linewidth": 1, "edgecolor": "white"}
    )
    plt.ylabel("")
    plt.title(f"Country Distribution - {label}")
    plt.tight_layout()
    filename = label
    plt.savefig(
        os.path.join(fig_dir, f"{filename}_country_dist_piechart.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.savefig(
        os.path.join(fig_dir, f"{filename}_country_dist_piechart.svg"),
        bbox_inches="tight"
    )
    plt.close()

# Overall cohort
plot_country_pie(
    df0,
    label="Overall cohort",
    fig_dir=fig_dir
)
# Per heterogeneity type
for label, data in df0.groupby("Heterogeneity Type"):
    plot_country_pie(
        data,
        label=label,
        fig_dir=fig_dir
    )

# --- Treatment history ---
def treatment_history_counts(data):
    counts = data["Cohort_Naive_b"].value_counts(dropna=False)
    return pd.Series({
        "treatment_naive": counts.get(1, 0),
        "not_treatment_naive": counts.get(0, 0),
        "NR_treatment_naive": counts.get("NR", 0),
        "missing_treatment_naive": data["Cohort_Naive_b"].isna().sum()
    })

# Overall cohort
treatment_overall = treatment_history_counts(df0)
results["Treatment history overall"] = treatment_overall
# Per heterogeneity type
treatment_split = (
    df0
    .groupby("Heterogeneity Type")
    .apply(treatment_history_counts, include_groups=False)
)

results["Treatment history split"] = treatment_split

# Gender distribution
female_col = "Females (n)"
male_col = "males_(n)_\ncomputed"
female_pct_col = "Females (%) \ncomputed"
mf_ratio_col = "male:1female_ratio_\ncomputed"

def to_numeric_clean(series):
    return pd.to_numeric(
        series.astype("string").replace(["NR", "n.a", "n.a.", "", " "], pd.NA),
        errors="coerce"
    )

def gender_distribution(data):
    females = to_numeric_clean(data[female_col])
    males = to_numeric_clean(data[male_col])
    female_pct_reported = to_numeric_clean(data[female_pct_col])
    mf_ratio = to_numeric_clean(data[mf_ratio_col])

    total_females = females.sum(skipna=True)
    total_males = males.sum(skipna=True)
    total_known = total_females + total_males

    female_pct = (total_females / total_known) * 100 if total_known > 0 else pd.NA
    male_pct = (total_males / total_known) * 100 if total_known > 0 else pd.NA

    return pd.Series({
        "female_n": int(total_females),
        "male_n": int(total_males),
        "total_with_sex_reported": int(total_known),
        "female_pct": female_pct,
        "male_pct": male_pct,
        "median_female_pct_across_cohorts": female_pct_reported.median(),
        "median_male_to_female_ratio": mf_ratio.median(),
        "n_cohorts_with_gender_data": pd.concat([females, males], axis=1).dropna().shape[0]
    })
    
gender_overall = gender_distribution(df0)
results["Gender distribution overall"] = gender_overall

gender_split = (
    df0
    .groupby("Heterogeneity Type")
    .apply(gender_distribution, include_groups=False)
)
results["Gender distribution split"] = gender_split

# --- Age distribution ---
age_col = "Cohort Age (median or mean)"
age_range_col = "Corhort Age (range)"  # excel spelling is wrong lol

def age_distribution(data):
    ages = pd.to_numeric(
        data[age_col].astype("string").replace(["NR", "n.a", "n.a.", "", " "], pd.NA),
        errors="coerce"
    )

    age_ranges = (
        data[age_range_col]
        .astype(str)
        .str.extract(r"(?P<age_min>\d+(?:\.\d+)?)\s*[-–]\s*(?P<age_max>\d+(?:\.\d+)?)") #capture this part and name it minimum_age \ one or more digits
        .apply(pd.to_numeric, errors="coerce")
    )

    valid_ranges = age_ranges.dropna()

    if valid_ranges.empty:
        overall_age_range = "NR"
        age_min = pd.NA
        age_max = pd.NA
    else:
        age_min = valid_ranges["age_min"].min()
        age_max = valid_ranges["age_max"].max()
        overall_age_range = f"{age_min:.0f} - {age_max:.0f}"

    return pd.Series({
        "median_age_across_cohorts": ages.median(),
        "age_min": age_min,
        "age_max": age_max,
        "age_range": overall_age_range,
        "n_cohorts_with_median_age": ages.notna().sum(),
        "n_cohorts_with_age_range": len(valid_ranges)
    })

age_overall = age_distribution(df0)
results["Age distribution overall"] = age_overall

age_split = (
    df0
    .groupby("Heterogeneity Type")
    .apply(age_distribution, include_groups=False)
)
results["Age distribution split"] = age_split

# --- Assay Category
assay_col = "assay_cate_new"

def assay_category_counts(data):
    assay = data[assay_col].replace({
        1: "RxDx",
        0: "LDT",
        "1": "RxDx",
        "0": "LDT"
    })

    counts = assay.value_counts(dropna=False)

    return pd.Series({
        "RxDx": counts.get("RxDx", 1),
        "LDT": counts.get("LDT", 0),
        "NR": counts.get("NR", pd.NA),
    })
assay_overall = assay_category_counts(df0)
results["Assay category overall"] = assay_overall

assay_split = (
    df0
    .groupby("Heterogeneity Type")
    .apply(assay_category_counts, include_groups=False)
)

results["Assay category split"] = assay_split


# --- Tumour primary origin: number of studies per tumour type ---

tumour_origin_col = "Tumor primary origin cohort specific (organ)"
study_id_col = "article nr"

tumour_type_study_counts = (
    df0
    .assign(
        tumour_type=df0[tumour_origin_col]
        .astype("string")
        .str.strip()
        .replace(["", "NR", "n.a", "n.a.", "nan"], pd.NA)
        .fillna("NR")
    )
    .groupby("tumour_type")[study_id_col]
    .nunique()
    .sort_values(ascending=False)
)

results["Tumour type study counts"] = tumour_type_study_counts

tumour_type_study_counts_split = (
    df0
    .assign(
        tumour_type=df0[tumour_origin_col]
        .astype("string")
        .str.strip()
        .replace(["", "NR", "n.a", "n.a.", "nan"], pd.NA)
        .fillna("NR")
    )
    .groupby(["Heterogeneity Type", "tumour_type"])[study_id_col]
    .nunique()
    .reset_index(name="n_studies")
    .sort_values(["Heterogeneity Type", "n_studies"], ascending=[True, False])
)

results["Tumour type study counts split"] = tumour_type_study_counts_split

# save descriptives table
output_file = os.path.join(fig_dir, "descriptive_results(ALL_TYPES).xlsx")

with pd.ExcelWriter(output_file) as writer:
    for key, value in results.items():
        if isinstance(value, pd.DataFrame):
            df_out = value
        elif isinstance(value, pd.Series):
            df_out = value.reset_index()
            df_out.columns = ["Variable", "Value"]
        else:
            df_out = pd.DataFrame({
                "Variable": [key],
                "Value": [value]
            })

        sheet_name = str(key)[:31]
        df_out.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Results saved to: {output_file}")

## Meta-analysis

In [ ]:
def pooled_disc_and_forest_funnel_plots(label='Overall'):
    
    output_dir = r"PATH/FOLDER"
    os.makedirs(output_dir, exist_ok=True)
    
    data = df0.copy()
    
    # Basic setup
    data["Category"] = data["Heterogeneity Type"]
    
    data["n_samplesize"] = pd.to_numeric(
        data["combined_sample_size_used_for_discordance"],
        errors="coerce"
    )
    
    data["n_discordant"] = pd.to_numeric(
        data["Combined_col_intra_inter discordance rate(n)"],
        errors="coerce"
    )
    
    data = data.dropna(subset=["Category", "n_samplesize", "n_discordant"]).copy()
    
    data["n_samplesize"] = data["n_samplesize"].astype(int)
    data["n_discordant"] = data["n_discordant"].astype(int)
    data["Percentage"] = data["n_discordant"] / data["n_samplesize"]
    
    # ---------------------------------------------------------------------------
    # Meta-analysis function (logit + Wilson optional export)
    # ---------------------------------------------------------------------------
    
    def perform_meta_analysis(subset, category):
        """
        Harmonized random-effects meta-analysis of proportions:
          - Study-level and pooled estimates computed on the logit scale; a 0.5 continuity correction is applied only to proportions of 0 or 1
          - Random-effects pooling via Paule–Mandel (method_re="iterated")
          - Hartung–Knapp–Sidik–Jonkman (HKSJ) CIs for pooled random-effects estimates
          - Wilson CIs exported for reference (not used for pooling)
        """
        # Counts
        x = subset['n_discordant'].to_numpy(dtype=float)
        n = subset['n_samplesize'].to_numpy(dtype=float)
    
        # ---------- Study-level point estimates ----------
        p_raw = x / n
    
        # ---------- Optional: Wilson CIs (for export only) ----------
        wilson_lo, wilson_hi = proportion_confint(
            count=x.astype(int), nobs=n.astype(int), alpha=0.05, method="wilson"
        )
        wilson_lo = np.clip(wilson_lo, 0, 1)
        wilson_hi = np.clip(wilson_hi, 0, 1)
    
        # ---------- Logit transform with boundary-only continuity correction ----------
        # Add 0.5 to the event and non-event counts only when x = 0 or x = n,
        # because the uncorrected logit and its sampling variance are then undefined.
        if np.any(n <= 0):
            raise ValueError("All sample sizes must be positive.")
        if np.any((x < 0) | (x > n)):
            raise ValueError("Discordant counts must satisfy 0 <= x <= n.")

        boundary_mask = (x == 0) | (x == n)
        x_adj = x.copy()
        n_adj = n.copy()
        x_adj[boundary_mask] += 0.5
        n_adj[boundary_mask] += 1.0
        p_adj = x_adj / n_adj
    
        y = logit(p_adj)
        v = 1.0/x_adj + 1.0/(n_adj - x_adj)
        se = np.sqrt(v)
    
        # ---------- Logit-based study-level CIs (display only) ----------
        ci_lo_logit = expit(y - 1.96 * se)
        ci_hi_logit = expit(y + 1.96 * se)
    
        # ---------- Random-effects pooling on logit scale ----------
        # Paule–Mandel estimator for tau² + HKSJ CIs
        res = combine_effects(y, v, method_re="iterated", use_t=True)
        use_t_flag = True
    
        mu_hat = res.mean_effect_re  # pooled *logit*
    
        # conf_int returns:
        # 0: FE (scale=1), 1: RE (scale=1), 2: FE + HKSJ, 3: RE + HKSJ
        ci_fe, ci_re, ci_fe_hksj, ci_re_hksj = res.conf_int(alpha=0.05, use_t=True)
        ci_re_lo, ci_re_hi = ci_re_hksj  # random-effects + HKSJ
    
        # back-transform pooled to %
        pooled_pct    = float(np.clip(expit(mu_hat)   * 100.0, 0, 100))
        pooled_lo_pct = float(np.clip(expit(ci_re_lo) * 100.0, 0, 100))
        pooled_hi_pct = float(np.clip(expit(ci_re_hi) * 100.0, 0, 100))
    
        # ---------- Heterogeneity ----------
        hom = res.test_homogeneity()
        q, df, p_het = hom.statistic, hom.df, hom.pvalue
        tau2 = res.tau2
        I2 = max(0, (q - df) / q) * 100.0 if q > 0 else 0.0
    
        # ---------- Prediction interval (logit scale) ----------
        k = len(y)
        df_pred = max(k - 2, 1)  # conservative df for PI
    
        # HKSJ variance for the pooled RE effect:
        se_pooled_hksj = np.sqrt(res.var_hksj_re)
    
        # Prediction SE includes between-study variance:
        se_pred = np.sqrt(se_pooled_hksj**2 + tau2)
        t_crit = stats.t.isf(0.025, df_pred)
    
        pi_lo_logit = mu_hat - t_crit * se_pred
        pi_hi_logit = mu_hat + t_crit * se_pred
    
        pi_lo_pct = float(np.clip(expit(pi_lo_logit) * 100.0, 0, 100))
        pi_hi_pct = float(np.clip(expit(pi_hi_logit) * 100.0, 0, 100))
    
        # ---------- Egger’s regression test (logit scale) ----------
        z = y / se
        X = add_constant(1.0 / se)
        egger_model = OLS(z, X).fit()
        egger_p = egger_model.pvalues[0]
    
        # ---------- DataFrames for export ----------
        study_level_df = pd.DataFrame({
            'Category': category,
            'Study': subset['Study'].values,
            'Sample Size': n.astype(int),
            'Events': x.astype(int),
            'Effect Size (proportion)': p_raw,
            'CI Lower (Logit 95%)': ci_lo_logit,
            'CI Upper (Logit 95%)': ci_hi_logit,
            'CI Lower (Wilson 95%)': wilson_lo,
            'CI Upper (Wilson 95%)': wilson_hi,
            'Effect (logit)': y,
            'SE (logit)': se
        })
    
        meta_summary_df = pd.DataFrame({
            'Category': [category],
            'Pooled % (Random Effects, logit-normal)': [pooled_pct],
            'CI Lower % (95%)': [pooled_lo_pct],
            'CI Upper % (95%)': [pooled_hi_pct],
            'Prediction Interval Lower % (95%)': [pi_lo_pct],
            'Prediction Interval Upper % (95%)': [pi_hi_pct],
            'Between-Study Variance (tau²)': [tau2],
            'Q-Statistic': [q],
            'df': [df],
            'p-value for Heterogeneity': [p_het],
            'I² (%)': [I2],
            'Knapp–Hartung used': [use_t_flag],
            'Egger intercept p-value (logit)': [egger_p],
            'Total Sample Size': [int(n.sum())],
            'Total Events': [int(x.sum())],
        })
    
        # ---------- plotting ----------
        class PlotResults:
            pass
    
        out = PlotResults()
        out.eff = p_raw
        out.row_ci_lower = ci_lo_logit
        out.row_ci_upper = ci_hi_logit
        out.study_names = subset['Study'].values
        out.n_samplesize = n.astype(int)
        out.n_discordant = x.astype(int)
        out.total_samplesize = int(n.sum())
        out.total_n_discordant = int(x.sum())
        out.mean_effect_re = pooled_pct / 100.0
        out._pooled_ci = (pooled_lo_pct / 100.0, pooled_hi_pct / 100.0)
        out.y = y
        out.v = v
        out.mu_hat_logit = mu_hat  # needed to center funnel plot
    
        return meta_summary_df, out, study_level_df
    
    # ---------------------------------------------------------------------------
    # Run all analyses
    # ---------------------------------------------------------------------------
    overall_results, overall_results_obj, overall_study_df = perform_meta_analysis(data, "Overall")
    grouped_results, grouped_study_dfs, grouped_results_objs = [], [], {}
    
    for category in data['Category'].unique():
        subset = data[data['Category'] == category]
        result_df, result_obj, study_df = perform_meta_analysis(subset, category)
        grouped_results.append(result_df)
        grouped_study_dfs.append(study_df)
        grouped_results_objs[category] = result_obj

        all_results_df = pd.concat([overall_results] + grouped_results, ignore_index=True)
        all_study_level_df = pd.concat([overall_study_df] + grouped_study_dfs, ignore_index=True)
    
        all_results_df.to_excel(
            os.path.join(output_dir, "meta_analysis_summary_overall_and_by_heterogeneity(ALL_TYPES).xlsx"),
            index=False
        )
        
        all_study_level_df.to_excel(
            os.path.join(output_dir, "meta_analysis_study_level_overall_and_by_heterogeneity(ALL_TYPES).xlsx"),
            index=False
        )
    
    # ---------------------------------------------------------------------------
    # Forest Plot
    # ---------------------------------------------------------------------------
    def plot_forest_custom(meta_result, category, save=False):
        eff = np.asarray(meta_result.eff)
        ci_lower = np.asarray(meta_result.row_ci_lower)
        ci_upper = np.asarray(meta_result.row_ci_upper)
        study_names = np.asarray(meta_result.study_names)
        n_samples = np.asarray(meta_result.n_samplesize)
        n_events = np.asarray(meta_result.n_discordant)
    
        n = len(eff)
        y_studies = np.arange(n)
        y_pooled = n + 0.5
    
        pooled_effect = meta_result.mean_effect_re
        pooled_lower, pooled_upper = meta_result._pooled_ci
    
        eff_pct = np.clip(eff * 100.0, 0.0, 100.0)
        ci_lower_pct = np.clip(ci_lower * 100.0, 0.0, 100.0)
        ci_upper_pct = np.clip(ci_upper * 100.0, 0.0, 100.0)
        pooled_effect_pct = np.clip(pooled_effect * 100.0, 0.0, 100.0)
        pooled_lower_pct = np.clip(pooled_lower * 100.0, 0.0, 100.0)
        pooled_upper_pct = np.clip(pooled_upper * 100.0, 0.0, 100.0)
        
        # Prevent negative error bars when adjusted CIs cross raw proportions
        lower_error = np.maximum(eff_pct - ci_lower_pct, 0)
        upper_error = np.maximum(ci_upper_pct - eff_pct, 0)
        
        pooled_lower_error = max(pooled_effect_pct - pooled_lower_pct, 0)
        pooled_upper_error = max(pooled_upper_pct - pooled_effect_pct, 0)
            
        def fmt_pct(x):
            if abs(x) < 0.05:
                x = 0.0
            return f"{x:.1f}"
    
        fig_height = (n + 5) * 0.5
        fig, ax = plt.subplots(figsize=(16, fig_height))
        study_name_x, sample_size_x, event_x, event_rate_x = -180, -70, -30, 140
        ax.set_xlim(-180, 160)
    
        header_y = -1
        ax.text(study_name_x, header_y, "Study", ha='left', fontweight='bold', fontsize=16)
        ax.text(sample_size_x, header_y, "Sample size", ha='center', fontweight='bold', fontsize=16)
        ax.text(event_x, header_y, "Event", ha='center', fontweight='bold', fontsize=16)
        ax.text(60, header_y, "Discordance percentage", ha='center', fontweight='bold', fontsize=16)
        ax.text(event_rate_x, header_y, "Event rate (95% CI)", ha='left', fontweight='bold', fontsize=16)
    
        for i in range(n):
            ax.errorbar(
                eff_pct[i], y_studies[i],
                xerr=[[lower_error[i]], [upper_error[i]]],
                fmt='o', color='black', capsize=4
            )
            ax.text(study_name_x, y_studies[i], str(study_names[i]), ha='left', va='center', fontsize=16)
            ax.text(sample_size_x, y_studies[i], str(n_samples[i]), ha='center', va='center', fontsize=16)
            ax.text(event_x, y_studies[i], str(n_events[i]), ha='center', va='center', fontsize=16)
            ax.text(
                event_rate_x, y_studies[i],
                f"{fmt_pct(eff_pct[i])}% [{fmt_pct(ci_lower_pct[i])} - {fmt_pct(ci_upper_pct[i])}]",
                ha='left', va='center', fontsize=16
            )
    
        # Pooled line
        ax.errorbar(
            pooled_effect_pct, y_pooled,
            xerr=[[pooled_lower_error], [pooled_upper_error]],
            fmt='s', color='red', capsize=4
        )
        pooled_label = "Pooled percentage\n(random effects)"
        ax.text(study_name_x, y_pooled, pooled_label, ha='left', va='center', fontweight='bold', fontsize=16)
        ax.text(
            sample_size_x, y_pooled, str(meta_result.total_samplesize),
            ha='center', va='center', fontweight='bold', fontsize=16
        )
        ax.text(
            event_rate_x, y_pooled,
            f"{fmt_pct(pooled_effect_pct)}% [{fmt_pct(pooled_lower_pct)} - {fmt_pct(pooled_upper_pct)}]",
            ha='left', va='center', fontweight='bold', fontsize=16
        )
    
        ax.set_ylim(-1.5, n + 1.5)
        ax.invert_yaxis()
        ax.set_xticks([0, 20, 40, 60, 80, 100])
        ax.set_xlabel("Discordance percentage (%)", fontsize=16)
    
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(left=False, labelleft=False, bottom=True)
        plt.tight_layout()
    
        if save:
            plt.savefig(os.path.join(output_dir, f"forest_{category}(ALL_TYPES).png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(output_dir, f"forest_{category}(ALL_TYPES).svg"), dpi=300, bbox_inches="tight")
        plt.show()
    
    # ---------------------------------------------------------------------------
    # Funnel Plot (logit scale, centered at pooled logit)
    # ---------------------------------------------------------------------------
    def plot_funnel(meta_result, category, save=False):
        """
        Funnel on the logit scale: study effects (y) vs SE, centered at pooled logit.
        Draws 95% funnel bounds (mu ± 1.96*SE).
        """
        y = np.asarray(meta_result.y)                 # logit effects
        se = np.sqrt(np.asarray(meta_result.v))      # SE (logit)
        mu = float(getattr(meta_result, "mu_hat_logit", np.nan))
    
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(y, se, alpha=0.7)
    
        # 95% 'funnel' triangle
        if np.isfinite(mu) and np.isfinite(se).any():
            max_se = np.nanmax(se)
            grid = np.linspace(0.0, max_se, 200)
            ax.plot(mu - 1.96*grid, grid, 'k--', linewidth=1)
            ax.plot(mu + 1.96*grid, grid, 'k--', linewidth=1)
            ax.axvline(mu, color='gray', linestyle='solid', linewidth=1)
    
        ax.invert_yaxis()  # conventional funnel
        ax.set_xlabel("Effect (logit proportion)", fontsize=12, fontweight='bold')
        ax.set_ylabel("Standard Error (logit)", fontsize=12, fontweight='bold')
    
        fig.suptitle(f'Funnel Plot — {category}', fontsize=14, fontweight='bold', y=1.02)
    
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
        plt.tight_layout()
        if save:
            plt.savefig(os.path.join(output_dir, f"funnel_{category}(ALL_TYPES).png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(output_dir, f"funnel_{category}(ALL_TYPES).svg"), dpi=300, bbox_inches="tight")
        plt.show()
    
    # ---------------------------------------------------------------------------
    # Run and Save Figures
    # ---------------------------------------------------------------------------
    # Overall
    plot_forest_custom(overall_results_obj, "Overall", save=True)
    plot_funnel(overall_results_obj, "Overall", save=True)
    
    # Each category
    for category, result_obj in grouped_results_objs.items():
        plot_forest_custom(result_obj, category, save=True)
        plot_funnel(result_obj, category, save=True)

pooled_results = pooled_disc_and_forest_funnel_plots()

## Stratified

In [ ]:
"""
Harmonized random-effects meta-analysis of proportions:
  - Study-level and pooled estimates computed on the logit scale; a 0.5 continuity correction is applied only to proportions of 0 or 1
  - Random-effects pooling via Paule–Mandel (method_re="iterated")
  - Hartung–Knapp–Sidik–Jonkman (HKSJ) CIs for pooled random-effects estimates
  - Wilson CIs exported for reference (not used for pooling)
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.meta_analysis import combine_effects
from statsmodels.api import OLS, add_constant
from statsmodels.stats.proportion import proportion_confint
from scipy import stats
from scipy.special import logit, expit
from IPython.display import display
from scipy.stats import norm
from scipy.stats import chi2
import itertools

def run_pooled_disc_and_stratifications():

    # ---------------------------------------------------------------------------
    # 1) Setup and Data Reading
    # ---------------------------------------------------------------------------
    sns.set_theme(style="white")
    
    output_dir = r"PATH/FOLDER"
    os.makedirs(output_dir, exist_ok=True)
       
    data = df0.copy() # (ALL_TYPES)
    
    # Basic columns
    data['n_samplesize'] = data['combined_sample_size_used_for_discordance'].astype(int)
    data['Category'] = data['Heterogeneity Type']          # expected: intra, inter, intra-inter
    data['n_discordant'] = data['Combined_col_intra_inter discordance rate(n)'].astype(int)

    # ---------------------------------------------------------------------------
    # Meta-analysis function (logit + Wilson optional export)
    # ---------------------------------------------------------------------------
    def perform_meta_analysis(subset, category):
    
        # Counts
        x = subset['n_discordant'].to_numpy(dtype=float)
        n = subset['n_samplesize'].to_numpy(dtype=float) 
    
        # ---------- Study-level point estimates ----------
        p_raw = x / n # the discordance rate, but not % yet.
    
        # ---------- Wilson CIs on the raw proportion x/n ----------
        wilson_lo, wilson_hi = proportion_confint(
            count=x.astype(int), nobs=n.astype(int), alpha=0.05, method="wilson" # alpha sets CI to 95%
        )
        wilson_lo = np.clip(wilson_lo, 0, 1) # lower bound: clipped between 0 and 1
        wilson_hi = np.clip(wilson_hi, 0, 1) # higher bound: clipped between 0 and 1
    
        # ---------- Logit transform with boundary-only continuity correction ----------
        # Add 0.5 to the event and non-event counts only when x = 0 or x = n,
        # because the uncorrected logit and its sampling variance are then undefined.
        if np.any(n <= 0):
            raise ValueError("All sample sizes must be positive.")
        if np.any((x < 0) | (x > n)):
            raise ValueError("Discordant counts must satisfy 0 <= x <= n.")

        boundary_mask = (x == 0) | (x == n)
        x_adj = x.copy()
        n_adj = n.copy()
        x_adj[boundary_mask] += 0.5
        n_adj[boundary_mask] += 1.0
        p_adj = x_adj / n_adj
    
        y = logit(p_adj) # 
        v = 1.0/x_adj + 1.0/(n_adj - x_adj) 
        se = np.sqrt(v) 
    
        # ---------- Logit-based study-level CIs ----------
        ci_lo_logit = expit(y - 1.96 * se) 
        ci_hi_logit = expit(y + 1.96 * se)
    
        # ---------- Random-effects pooling on logit scale proportions ----------

            # Paule–Mandel estimator for tau² + HKSJ CIs
        res = combine_effects(y, v, method_re="iterated", use_t=True)
        use_t_flag = True
    
        mu_hat = res.mean_effect_re  # pooled *logit* ## calling for pooled effect (logit scale)
    
        # conf_int returns:
        # 0: FE (scale=1), 1: RE (scale=1), 2: FE + HKSJ, 3: RE + HKSJ
        ci_fe, ci_re, ci_fe_hksj, ci_re_hksj = res.conf_int(alpha=0.05, use_t=True)
        ci_re_lo, ci_re_hi = ci_re_hksj  # random-effects + HKSJ-adjustment. 
    
        '''
        Random-effects model: Each study may estimate a different true effect. 
        Effects vary across studies due to both sampling error and heterogeneity.
        --> allows for real differences between studies and is more realistic when heterogeneity exists.
        Use with the KJSH adjustment, because it adjusts the CI for small-study bias and extra uncertainty
    
        Fixed-effects model: All studies share one true effect. 
        Differences in results are due only to sampling error.
        
        '''
    
        # back-transform pooled to %
        pooled_pct    = float(np.clip(expit(mu_hat)   * 100.0, 0, 100)) 
        pooled_lo_pct = float(np.clip(expit(ci_re_lo) * 100.0, 0, 100))
        pooled_hi_pct = float(np.clip(expit(ci_re_hi) * 100.0, 0, 100))
    
        # ---------- Heterogeneity ----------Cochran’s Q test. 
        hom = res.test_homogeneity()
        q, df, p_het = hom.statistic, hom.df, hom.pvalue
        tau2 = res.tau2
        I2 = max(0, (q - df) / q) * 100.0 if q > 0 else 0.0 # 
    
        # ---------- Prediction interval (logit scale) ----------
        
        k = len(y) 
        df_pred = max(k - 2, 1)  
    
        # HKSJ variance for the pooled RE effect:
        se_pooled_hksj = np.sqrt(res.var_hksj_re) 
    
        # Prediction SE includes between-study variance:
        se_pred = np.sqrt(se_pooled_hksj**2 + tau2) 
        t_crit = stats.t.isf(0.025, df_pred) 
    
        pi_lo_logit = mu_hat - t_crit * se_pred 
        pi_hi_logit = mu_hat + t_crit * se_pred
    
        pi_lo_pct = float(np.clip(expit(pi_lo_logit) * 100.0, 0, 100)) 
        pi_hi_pct = float(np.clip(expit(pi_hi_logit) * 100.0, 0, 100))
    
        # ---------- Egger’s regression test (logit scale) ---------- Publication Bias. 
        z = y / se 
        X = add_constant(1.0 / se) 
        egger_model = OLS(z, X).fit() 
        egger_p = egger_model.pvalues[0] 
    
        # ---------- DataFrames for export ---------- NOT POOLED --- just individual study effect-sizes and CIs.
        study_level_df = pd.DataFrame({
            'Category': category,
            'Study': subset['Study'].values,
            'Sample Size': n.astype(int),
            'Events': x.astype(int),
            'Effect Size (proportion)': p_raw,
            'CI Lower (Logit 95%)': ci_lo_logit,    # of the logit transformed proportion values
            'CI Upper (Logit 95%)': ci_hi_logit,    # of the logit transformed proportion values
            'CI Lower (Wilson 95%)': wilson_lo,     # of the raw proportion values
            'CI Upper (Wilson 95%)': wilson_hi,     # of the raw proportion values
            'Effect (logit)': y,
            'SE (logit)': se
        })
    
        meta_summary_df = pd.DataFrame({
            'Category': [category],
            'Pooled % (Random Effects, logit-normal)': [pooled_pct],
            'CI Lower % (95%)': [pooled_lo_pct],
            'CI Upper % (95%)': [pooled_hi_pct],
            'Prediction Interval Lower % (95%)': [pi_lo_pct],
            'Prediction Interval Upper % (95%)': [pi_hi_pct],
            'Between-Study Variance (tau²)': [tau2],
            'Q-Statistic': [q],
            'df': [df],
            'p-value for Heterogeneity': [p_het],
            'I² (%)': [I2],
            'Knapp–Hartung used': [use_t_flag],
            'Egger intercept p-value (logit)': [egger_p],
            'Total Sample Size': [int(n.sum())],
            'Total Events': [int(x.sum())],
        })
    
        # ---------- plotting ----------
        class PlotResults:
            pass
    
        out = PlotResults()
        out.eff = p_raw
        out.row_ci_lower = ci_lo_logit
        out.row_ci_upper = ci_hi_logit
        out.study_names = subset['Study'].values
        out.n_samplesize = n.astype(int)
        out.n_discordant = x.astype(int)
        out.total_samplesize = int(n.sum())
        out.total_n_discordant = int(x.sum())
        out.mean_effect_re = pooled_pct / 100.0
        out._pooled_ci = (pooled_lo_pct / 100.0, pooled_hi_pct / 100.0)
        out.y = y
        out.v = v
        out.mu_hat_logit = mu_hat
        out.method_label = "RE + HKSJ"
        out.q = q
        out.df = df
        out.tau2 = tau2
        out.p_het = p_het
    
        return meta_summary_df, out, study_level_df
    
    # ---------------------------------------------------------------------------
    # Run all analyses
    # ---------------------------------------------------------------------------
    overall_results, overall_results_obj, overall_study_df = perform_meta_analysis(data, "Overall")
    
    grouped_results, grouped_study_dfs, grouped_results_objs = [], [], {}
    for category in data['Category'].unique():
        subset = data[data['Category'] == category]
        result_df, result_obj, study_df = perform_meta_analysis(subset, category)
        grouped_results.append(result_df)
        grouped_study_dfs.append(study_df)
        grouped_results_objs[category] = result_obj
    
    all_results_df = pd.concat([overall_results] + grouped_results, ignore_index=True)
    all_study_level_df = pd.concat([overall_study_df] + grouped_study_dfs, ignore_index=True)
    
    # Save tables
    all_results_df.to_excel(os.path.join(output_dir, f"meta_analysis_summary(ALL_TYPES).xlsx"), index=False)
    display(all_results_df)
    
    # ---------------------------------------------------------------------------
    # Forest Plot
    # ---------------------------------------------------------------------------
    def plot_forest_custom(meta_result, category, save=False):
        eff = np.asarray(meta_result.eff)
        ci_lower = np.asarray(meta_result.row_ci_lower)
        ci_upper = np.asarray(meta_result.row_ci_upper)
        study_names = np.asarray(meta_result.study_names)
        n_samples = np.asarray(meta_result.n_samplesize)
        n_events = np.asarray(meta_result.n_discordant)
    
        n = len(eff)
        y_studies = np.arange(n)
        y_pooled = n + 0.5
    
        pooled_effect = meta_result.mean_effect_re
        pooled_lower, pooled_upper = meta_result._pooled_ci
    
        eff_pct = np.clip(eff * 100.0, 0.0, 100.0)
        ci_lower_pct = np.clip(ci_lower * 100.0, 0.0, 100.0)
        ci_upper_pct = np.clip(ci_upper * 100.0, 0.0, 100.0)
        pooled_effect_pct = np.clip(pooled_effect * 100.0, 0.0, 100.0)
        pooled_lower_pct = np.clip(pooled_lower * 100.0, 0.0, 100.0)
        pooled_upper_pct = np.clip(pooled_upper * 100.0, 0.0, 100.0)

        # Prevent negative error bars when adjusted CIs cross raw proportions
        lower_error = np.maximum(eff_pct - ci_lower_pct, 0)
        upper_error = np.maximum(ci_upper_pct - eff_pct, 0)
        
        pooled_lower_error = max(pooled_effect_pct - pooled_lower_pct, 0)
        pooled_upper_error = max(pooled_upper_pct - pooled_effect_pct, 0)
    
        def fmt_pct(x):
            if abs(x) < 0.05:
                x = 0.0
            return f"{x:.1f}"
    
        fig_height = (n + 5) * 0.5
        fig, ax = plt.subplots(figsize=(16, fig_height))
        study_name_x, sample_size_x, event_x, event_rate_x = -180, -70, -30, 140
        ax.set_xlim(-180, 160)
    
        header_y = -1
        ax.text(study_name_x, header_y, "Study", ha='left', fontweight='bold', fontsize=16)
        ax.text(sample_size_x, header_y, "Sample size", ha='center', fontweight='bold', fontsize=16)
        ax.text(event_x, header_y, "Event", ha='center', fontweight='bold', fontsize=16)
        ax.text(60, header_y, "Discordance percentage", ha='center', fontweight='bold', fontsize=16)
        ax.text(event_rate_x, header_y, "Event rate (95% CI)", ha='left', fontweight='bold', fontsize=16)
    
        for i in range(n):
            ax.errorbar(
                eff_pct[i], y_studies[i],
                #xerr=[[eff_pct[i] - ci_lower_pct[i]], [ci_upper_pct[i] - eff_pct[i]]],
                xerr=[[lower_error[i]], [upper_error[i]]],
                fmt='o', color='black', capsize=4
            )
            ax.text(study_name_x, y_studies[i], str(study_names[i]), ha='left', va='center', fontsize=16)
            ax.text(sample_size_x, y_studies[i], str(n_samples[i]), ha='center', va='center', fontsize=16)
            ax.text(event_x, y_studies[i], str(n_events[i]), ha='center', va='center', fontsize=16)
            ax.text(
                event_rate_x, y_studies[i],
                f"{fmt_pct(eff_pct[i])}% [{fmt_pct(ci_lower_pct[i])} - {fmt_pct(ci_upper_pct[i])}]",
                ha='left', va='center', fontsize=16
            )
    
        # Pooled line
        ax.errorbar(
            pooled_effect_pct, y_pooled,
            #xerr=[[pooled_effect_pct - pooled_lower_pct], [pooled_upper_pct - pooled_effect_pct]],
            xerr=[[pooled_lower_error], [pooled_upper_error]],
            fmt='s', color='red', capsize=4
        )
        pooled_label = "Pooled percentage\n(random effects)"
        ax.text(study_name_x, y_pooled, pooled_label, ha='left', va='center', fontweight='bold', fontsize=16)
        ax.text(
            sample_size_x, y_pooled, str(meta_result.total_samplesize),
            ha='center', va='center', fontweight='bold', fontsize=16
        )
        ax.text(
            event_rate_x, y_pooled,
            f"{fmt_pct(pooled_effect_pct)}% [{fmt_pct(pooled_lower_pct)} - {fmt_pct(pooled_upper_pct)}]",
            ha='left', va='center', fontweight='bold', fontsize=16
        )
    
        ax.set_ylim(-1.5, n + 1.5)
        ax.invert_yaxis()
        ax.set_xticks([0, 20, 40, 60, 80, 100])
        ax.set_xlabel("Discordance percentage (%)", fontsize=16)
    
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(left=False, labelleft=False, bottom=True)
        plt.tight_layout()
    
        if save:
            plt.savefig(os.path.join(output_dir, f"forest_{category}(ALL_TYPES).png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(output_dir, f"forest_{category}(ALL_TYPES).svg"), dpi=300, bbox_inches="tight")
        plt.show()
    
    # ---------------------------------------------------------------------------
    # Funnel Plot (logit scale, centered at pooled logit)
    # ---------------------------------------------------------------------------
    def plot_funnel(meta_result, category, save=False):
        """
        Funnel on the logit scale: study effects (y) vs SE, centered at pooled logit.
        Draws 95% funnel bounds (mu ± 1.96*SE).
        """
        y = np.asarray(meta_result.y)                 # logit effects
        se = np.sqrt(np.asarray(meta_result.v))       # SE (logit)
        mu = float(getattr(meta_result, "mu_hat_logit", np.nan))
    
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(y, se, alpha=0.7)
    
        # 95% 'funnel' triangle
        if np.isfinite(mu) and np.isfinite(se).any():
            max_se = np.nanmax(se)
            grid = np.linspace(0.0, max_se, 200)
            ax.plot(mu - 1.96*grid, grid, 'k--', linewidth=1)
            ax.plot(mu + 1.96*grid, grid, 'k--', linewidth=1)
            ax.axvline(mu, color='gray', linestyle='solid', linewidth=1)
    
        ax.invert_yaxis()  # conventional funnel
        ax.set_xlabel("Effect (logit proportion)", fontsize=12, fontweight='bold')
        ax.set_ylabel("Standard Error (logit)", fontsize=12, fontweight='bold')
        fig.suptitle(f'Funnel Plot — {category}', fontsize=14, fontweight='bold', y=1.02)
    
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
        plt.tight_layout()
        if save:
            plt.savefig(os.path.join(output_dir, f"funnel_{category}(ALL_TYPES).png"), dpi=300, bbox_inches="tight")
            plt.savefig(os.path.join(output_dir, f"funnel_{category}(ALL_TYPES).svg"), dpi=300, bbox_inches="tight")
        plt.show()
    
    # ---------------------------------------------------------------------------
    # Run and Save Figures
    # ---------------------------------------------------------------------------
    # Overall
    plot_forest_custom(overall_results_obj, "Overall", save=True)
    plot_funnel(overall_results_obj, "Overall", save=True)
    
    # Each category
    for category, result_obj in grouped_results_objs.items():
        plot_forest_custom(result_obj, category, save=True)
        plot_funnel(result_obj, category, save=True)
    
    # ###############################################################################
    # #              Additional Subgroup Meta‐Analyses (Stratified)              #
    # ###############################################################################
    
    all_summary_dfs = []
    all_results_objs = {}
    
    for category, category_data in data.groupby("Heterogeneity Type"):
    
        # Optional: add unstratified pooled result for this heterogeneity type
        summary_df, res_obj, study_df = perform_meta_analysis(
            category_data,
            f"{category} | Overall"
        )
    
        all_summary_dfs.append(summary_df)
        all_results_objs[f"{category} | Overall"] = res_obj
    
        # Stratifications within this heterogeneity type
        for col, info in STRATIFICATIONS.items():
    
            if col not in category_data.columns:
                print(f"{category} - {col}: column missing, skipped")
                continue
    
            mapping = info["mapping"]
            mode = info["mode"]
    
            if mode == "both":
                for val, label in mapping.items():
                    subset = category_data[category_data[col] == val]
    
                    if not subset.empty:
                        group_label = f"{category} | {col}: {label}"
    
                        summary_df, res_obj, study_df = perform_meta_analysis(
                            subset,
                            group_label
                        )
    
                        all_summary_dfs.append(summary_df)
                        all_results_objs[group_label] = res_obj
    
            elif mode == "only1":
                subset = category_data[category_data[col] == 1]
    
                if not subset.empty:
                    group_label = f"{category} | {col}: {mapping.get(1, '1')}"
    
                    summary_df, res_obj, study_df = perform_meta_analysis(
                        subset,
                        group_label
                    )
    
                    all_summary_dfs.append(summary_df)
                    all_results_objs[group_label] = res_obj

    ###############################################################################
    #                Combine and Save Summary Results                          #
    ###############################################################################
    all_results_df = pd.concat(all_summary_dfs, ignore_index=True)

    all_results_df.to_excel(os.path.join(output_dir, f"meta_analysis_summary_stratifications(ALL_TYPES).xlsx"), index=False)
    
    ###############################################################################
    #                Pairwise Statistical Comparisons DataFrame                #
    ###############################################################################

    def get_pooled_logit_and_se(result_obj):
        res = combine_effects(
            result_obj.y,
            result_obj.v,
            method_re="iterated",
            use_t=True
        )
        mu_logit = float(res.mean_effect_re)
        se_logit = float(res.sd_eff_w_re)
        return mu_logit, se_logit

    comparison_rows = []
 
    strat_groups = {}

    for key in all_results_objs.keys():
        if key == "Overall":
            continue
        if ":" in key:
            strat_var = key.split(":")[0].strip()
            strat_groups.setdefault(strat_var, []).append(key)
 
    for strat_var, keys in strat_groups.items():
        if len(keys) > 1:
            for key1, key2 in itertools.combinations(keys, 2):
                logit1, se1 = get_pooled_logit_and_se(all_results_objs[key1])
                logit2, se2 = get_pooled_logit_and_se(all_results_objs[key2])
                diff = logit1 - logit2
                se_diff = np.sqrt(se1**2 + se2**2)
                z_stat = diff / se_diff if se_diff > 0 else np.nan
                p_value = 2 * (1 - norm.cdf(abs(z_stat))) if se_diff > 0 else np.nan
 
                rate1 = float(all_results_objs[key1].mean_effect_re)
                rate2 = float(all_results_objs[key2].mean_effect_re)
 
                comparison_rows.append({
                    "Comparison": f"{key1} vs {key2}",
                    "Discordance rate 1": f"{rate1:.3f}",
                    "Discordance rate 2": f"{rate2:.3f}",
                    "Statistic": f"{z_stat:.3f}" if np.isfinite(z_stat) else "NA",
                    "p-value (z-test)": f"{p_value:.3f}" if np.isfinite(z_stat) else "NA",
                    "Method (RE)": all_results_objs[key1].method_label
                })
 
    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df)
    comparison_df.to_excel(os.path.join(output_dir, f"meta_analysis_pairwise_COMPARISONS(ALL_TYPES).xlsx"), index=False)

    return {
        "overall_obj": overall_results_obj,      # pooled overall
        "all_results_objs": all_results_objs,    # dict: "Overall", "Asian_Cohort: Asian", ...
        "all_results_df": all_results_df,        # summary table
        "comparison_df": comparison_df           # pairwise comparisons table
    }

pooled_strati = run_pooled_disc_and_stratifications()

all_results_objs = pooled_strati["all_results_objs"]
overall_results_obj = pooled_strati["overall_obj"]

In [ ]:
# Non-stratified 95% prediction intervals: gastric and cross-tumour intra/inter analyses only
import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import expit, logit
from statsmodels.stats.meta_analysis import combine_effects
from IPython.display import display

_PI_OUTCOMES = ("Intra-tumoural (T-T)", "Inter-tumoural (T-T)")


def _prepare_non_stratified_counts(cohort, outcome):
    """Return validated study-level discordant counts and denominators for one outcome."""
    subset = cohort.loc[cohort["Heterogeneity Type"] == outcome].copy()

    x = pd.to_numeric(
        subset["Combined_col_intra_inter discordance rate(n)"],
        errors="coerce",
    )
    n = pd.to_numeric(
        subset["combined_sample_size_used_for_discordance"],
        errors="coerce",
    )

    valid = x.notna() & n.notna()
    x = x.loc[valid].to_numpy(dtype=float)
    n = n.loc[valid].to_numpy(dtype=float)

    if len(x) < 3:
        raise ValueError(
            f"{outcome}: at least three studies are required for a prediction interval."
        )
    if np.any(n <= 0):
        raise ValueError(f"{outcome}: all sample sizes must be positive.")
    if np.any((x < 0) | (x > n)):
        raise ValueError(f"{outcome}: discordant counts must satisfy 0 <= x <= n.")
    if not np.allclose(x, np.round(x)) or not np.allclose(n, np.round(n)):
        raise ValueError(f"{outcome}: event counts and sample sizes must be integers.")

    return x, n


def _prediction_interval_row(cohort, scope, outcome):
    """
    Paule-Mandel random-effects pooling on the logit scale with HKSJ confidence
    intervals. A 0.5 continuity correction is used only for boundary proportions.
    The 95% prediction interval is calculated on the logit scale as
    mu_hat +/- t_(k-2) * sqrt(tau^2 + Var_HKSJ(mu_hat)).
    """
    x, n = _prepare_non_stratified_counts(cohort, outcome)

    boundary = (x == 0) | (x == n)
    x_adj = x.copy()
    n_adj = n.copy()
    x_adj[boundary] += 0.5
    n_adj[boundary] += 1.0

    p_adj = x_adj / n_adj
    y = logit(p_adj)
    v = 1.0 / x_adj + 1.0 / (n_adj - x_adj)

    result = combine_effects(y, v, method_re="iterated", use_t=True)
    mu_hat = float(result.mean_effect_re)
    tau2 = max(float(result.tau2), 0.0)

    _, _, _, ci_re_hksj = result.conf_int(alpha=0.05, use_t=True)
    ci_low_logit, ci_high_logit = map(float, ci_re_hksj)

    k = len(y)
    se_mu_hksj = np.sqrt(max(float(result.var_hksj_re), 0.0))
    prediction_se = np.sqrt(tau2 + se_mu_hksj**2)
    t_critical = float(stats.t.ppf(0.975, df=k - 2))

    pi_low_logit = mu_hat - t_critical * prediction_se
    pi_high_logit = mu_hat + t_critical * prediction_se

    q = float(result.q)
    i2 = max(0.0, (q - (k - 1)) / q) * 100.0 if q > 0 else 0.0

    return {
        "Scope": scope,
        "Outcome": outcome,
        "Studies": k,
        "Patients": int(n.sum()),
        "Discordant cases": int(x.sum()),
        "Pooled estimate (%)": 100.0 * float(expit(mu_hat)),
        "HKSJ CI lower (%)": 100.0 * float(expit(ci_low_logit)),
        "HKSJ CI upper (%)": 100.0 * float(expit(ci_high_logit)),
        "Prediction interval lower (%)": 100.0 * float(expit(pi_low_logit)),
        "Prediction interval upper (%)": 100.0 * float(expit(pi_high_logit)),
        "tau² (logit scale)": tau2,
        "I² (%)": i2,
        "PI degrees of freedom": k - 2,
    }


_prediction_interval_rows = []
for _scope, _cohort in (
    ("Gastric/oesophagogastric", gastric_only),
    ("Cross-tumour", df0),
):
    for _outcome in _PI_OUTCOMES:
        _prediction_interval_rows.append(
            _prediction_interval_row(_cohort, _scope, _outcome)
        )

prediction_intervals_primary_and_cross_tumour = pd.DataFrame(
    _prediction_interval_rows
)

display(
    prediction_intervals_primary_and_cross_tumour.round(
        {
            "Pooled estimate (%)": 2,
            "HKSJ CI lower (%)": 2,
            "HKSJ CI upper (%)": 2,
            "Prediction interval lower (%)": 2,
            "Prediction interval upper (%)": 2,
            "tau² (logit scale)": 3,
            "I² (%)": 1,
        }
    )
)


In [ ]:
# Binomial-normal GLMM sensitivity analysis: gastric and cross-tumour intra/inter analyses only
# Model: x_i ~ Binomial(n_i, p_i), logit(p_i) = mu + u_i, u_i ~ Normal(0, tau^2)
# The marginal likelihood is evaluated with adaptive Gauss-Hermite quadrature.
# No continuity correction is used. Confidence intervals for mu use profile likelihood.

import numpy as np
import pandas as pd
from numpy.polynomial.hermite import hermgauss
from scipy.optimize import brentq, minimize, minimize_scalar
from scipy.special import expit, gammaln, logit, logsumexp
from scipy.stats import chi2
from IPython.display import display

_GLMM_OUTCOMES = ("Intra-tumoural (T-T)", "Inter-tumoural (T-T)")


def _fit_binomial_normal_glmm(x, n, quadrature_points=15):
    """Fit an intercept-only binomial-normal random-effects GLMM by maximum likelihood."""
    x = np.asarray(x, dtype=float)
    n = np.asarray(n, dtype=float)

    if len(x) < 2:
        raise ValueError("At least two studies are required for the GLMM.")
    if np.any(n <= 0):
        raise ValueError("All sample sizes must be positive.")
    if np.any((x < 0) | (x > n)):
        raise ValueError("Discordant counts must satisfy 0 <= x <= n.")
    if not np.allclose(x, np.round(x)) or not np.allclose(n, np.round(n)):
        raise ValueError("Event counts and sample sizes must be integers.")

    gh_nodes, gh_weights = hermgauss(quadrature_points)
    log_gh_weights = np.log(gh_weights)
    log_binomial_coefficients = (
        gammaln(n + 1.0) - gammaln(x + 1.0) - gammaln(n - x + 1.0)
    )

    def _conditional_modes(mu, tau):
        """Newton-Raphson modes of each study-specific random effect."""
        tau2 = tau * tau

        # The empirical logit is used only as a numerical starting value.
        initial_p = (x + 0.5) / (n + 1.0)
        u = logit(initial_p) - mu

        for _ in range(100):
            eta = mu + u
            p = expit(eta)
            score = x - n * p - u / tau2
            curvature = n * p * (1.0 - p) + 1.0 / tau2
            step = score / curvature
            u_new = u + step

            if np.max(np.abs(step)) < 1e-10:
                u = u_new
                break
            u = u_new
        else:
            raise RuntimeError("Random-effect mode calculation did not converge.")

        eta = mu + u
        p = expit(eta)
        curvature = n * p * (1.0 - p) + 1.0 / tau2
        return u, curvature

    def _log_likelihood(parameters):
        mu = float(parameters[0])
        tau = float(np.exp(parameters[1]))

        modes, curvature = _conditional_modes(mu, tau)
        local_scale = np.sqrt(2.0 / curvature)

        random_effects = (
            modes[:, None] + local_scale[:, None] * gh_nodes[None, :]
        )
        eta = mu + random_effects

        log_kernel = (
            x[:, None] * eta
            - n[:, None] * np.logaddexp(0.0, eta)
            - random_effects**2 / (2.0 * tau**2)
        )

        # Adaptive Gauss-Hermite approximation to each marginal study likelihood.
        log_integral = (
            0.5 * np.log(2.0 / curvature)
            + logsumexp(
                log_gh_weights[None, :]
                + log_kernel
                + gh_nodes[None, :] ** 2,
                axis=1,
            )
        )

        log_marginal = (
            log_binomial_coefficients
            - 0.5 * np.log(2.0 * np.pi)
            - np.log(tau)
            + log_integral
        )
        return float(np.sum(log_marginal))

    def _negative_log_likelihood(parameters):
        try:
            value = _log_likelihood(parameters)
        except (FloatingPointError, RuntimeError):
            return np.inf
        return -value if np.isfinite(value) else np.inf

    pooled_start = (x.sum() + 0.5) / (n.sum() + 1.0)
    mu_start = float(logit(pooled_start))

    # Multiple starting values reduce sensitivity to local numerical solutions.
    candidate_fits = []
    for tau_start in (0.05, 0.20, 0.50, 1.00, 2.00):
        fit = minimize(
            _negative_log_likelihood,
            np.array([mu_start, np.log(tau_start)], dtype=float),
            method="L-BFGS-B",
            bounds=[(-15.0, 15.0), (-12.0, 3.0)],
            options={"maxiter": 2000, "ftol": 1e-12, "gtol": 1e-8},
        )
        if np.isfinite(fit.fun):
            candidate_fits.append(fit)

    if not candidate_fits:
        raise RuntimeError("GLMM optimisation failed for all starting values.")

    best_fit = min(candidate_fits, key=lambda result: result.fun)
    mu_hat = float(best_fit.x[0])
    tau_hat = float(np.exp(best_fit.x[1]))
    maximum_log_likelihood = -float(best_fit.fun)

    # Profile-likelihood confidence interval for the pooled logit.
    profile_cache = {}

    def _profile_log_likelihood(mu):
        cache_key = round(float(mu), 10)
        if cache_key in profile_cache:
            return profile_cache[cache_key]

        profiled_fit = minimize_scalar(
            lambda log_tau: -_log_likelihood([mu, log_tau]),
            bounds=(-12.0, 3.0),
            method="bounded",
            options={"xatol": 1e-8, "maxiter": 500},
        )
        if not profiled_fit.success or not np.isfinite(profiled_fit.fun):
            raise RuntimeError("Profile-likelihood optimisation failed.")

        profile_value = -float(profiled_fit.fun)
        profile_cache[cache_key] = profile_value
        return profile_value

    likelihood_ratio_cutoff = float(chi2.ppf(0.95, df=1))

    def _profile_root(mu):
        return (
            2.0
            * (maximum_log_likelihood - _profile_log_likelihood(mu))
            - likelihood_ratio_cutoff
        )

    def _find_profile_limit(direction):
        step = 0.25
        outer = mu_hat + direction * step

        while _profile_root(outer) < 0.0 and abs(outer) < 15.0:
            step *= 1.6
            outer = mu_hat + direction * step

        if _profile_root(outer) < 0.0:
            raise RuntimeError("Could not bracket a profile-likelihood limit.")

        lower, upper = (
            (outer, mu_hat) if direction < 0 else (mu_hat, outer)
        )
        return float(brentq(_profile_root, lower, upper, xtol=1e-8))

    ci_low_logit = _find_profile_limit(-1)
    ci_high_logit = _find_profile_limit(1)

    return {
        "mu_logit": mu_hat,
        "tau": tau_hat,
        "tau2": tau_hat**2,
        "pooled": float(expit(mu_hat)),
        "ci_low": float(expit(ci_low_logit)),
        "ci_high": float(expit(ci_high_logit)),
        "log_likelihood": maximum_log_likelihood,
        "converged": bool(best_fit.success),
        "optimizer_message": str(best_fit.message),
        "quadrature_points": quadrature_points,
    }


def _glmm_sensitivity_row(cohort, scope, outcome):
    subset = cohort.loc[cohort["Heterogeneity Type"] == outcome].copy()

    x = pd.to_numeric(
        subset["Combined_col_intra_inter discordance rate(n)"],
        errors="coerce",
    )
    n = pd.to_numeric(
        subset["combined_sample_size_used_for_discordance"],
        errors="coerce",
    )

    valid = x.notna() & n.notna()
    x = x.loc[valid].to_numpy(dtype=float)
    n = n.loc[valid].to_numpy(dtype=float)

    fit = _fit_binomial_normal_glmm(x, n, quadrature_points=15)

    return {
        "Scope": scope,
        "Outcome": outcome,
        "Studies": len(x),
        "Patients": int(n.sum()),
        "Discordant cases": int(x.sum()),
        "GLMM pooled estimate (%)": 100.0 * fit["pooled"],
        "Profile CI lower (%)": 100.0 * fit["ci_low"],
        "Profile CI upper (%)": 100.0 * fit["ci_high"],
        "tau² (logit scale)": fit["tau2"],
        "Log likelihood": fit["log_likelihood"],
        "Quadrature points": fit["quadrature_points"],
        "Converged": fit["converged"],
    }


_glmm_rows = []
for _scope, _cohort in (
    ("Gastric/oesophagogastric", gastric_only),
    ("Cross-tumour", df0),
):
    for _outcome in _GLMM_OUTCOMES:
        _glmm_rows.append(
            _glmm_sensitivity_row(_cohort, _scope, _outcome)
        )

binomial_normal_glmm_sensitivity_primary_and_cross_tumour = pd.DataFrame(
    _glmm_rows
)

if not binomial_normal_glmm_sensitivity_primary_and_cross_tumour[
    "Converged"
].all():
    raise RuntimeError("At least one GLMM did not converge.")

display(
    binomial_normal_glmm_sensitivity_primary_and_cross_tumour.round(
        {
            "GLMM pooled estimate (%)": 2,
            "Profile CI lower (%)": 2,
            "Profile CI upper (%)": 2,
            "tau² (logit scale)": 3,
            "Log likelihood": 3,
        }
    )
)
